Step 1 — Synthetic mathematical validation

In [1]:
# ============================================================
# CASCADE — STEP 1.1
# Environment, reproducibility, and experiment configuration
# ============================================================

import sys
import platform
import time
import math
import random
from dataclasses import dataclass
from itertools import product

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("=" * 80)
print("CASCADE — STEP 1.1: ENVIRONMENT")
print("=" * 80)

print(f"Python          : {sys.version.split()[0]}")
print(f"Platform        : {platform.platform()}")
print(f"NumPy           : {np.__version__}")
print(f"Pandas          : {pd.__version__}")
print(f"Random seed     : {SEED}")

# ------------------------------------------------------------
# Core experiment settings
# ------------------------------------------------------------

N_CONTROLS = 10

CAPABILITY_LEVELS = np.array([
    0.00,
    0.25,
    0.50,
    0.75,
    1.00
])

print(f"\nNumber controls : {N_CONTROLS}")
print(f"Capability lvls : {CAPABILITY_LEVELS.tolist()}")
print(f"Search space    : {len(CAPABILITY_LEVELS) ** N_CONTROLS:,} configurations")

print("\n[OK] Step 1.1 completed.")

CASCADE — STEP 1.1: ENVIRONMENT
Python          : 3.13.15
Platform        : Linux-6.6.122+-x86_64-with-glibc2.39
NumPy           : 2.1.3
Pandas          : 2.2.3
Random seed     : 42

Number controls : 10
Capability lvls : [0.0, 0.25, 0.5, 0.75, 1.0]
Search space    : 9,765,625 configurations

[OK] Step 1.1 completed.


Step 1.2 — Define the synthetic defensive architecture

In [2]:
# ============================================================
# CASCADE — STEP 1.2
# Synthetic defensive capabilities
# ============================================================

controls = pd.DataFrame({
    "id": [
        "C1", "C2", "C3", "C4", "C5",
        "C6", "C7", "C8", "C9", "C10"
    ],

    "name": [
        "Telemetry",
        "Security Logging",
        "Network Monitoring",
        "SIEM Analysis",
        "Automated Response",
        "Network Segmentation",
        "Firewall Filtering",
        "Rate Limiting",
        "Access Control",
        "Backup Recovery"
    ],

    # Resource cost at full operation.
    # Synthetic for Step 1 only.
    "resource_cost": [
        8, 10, 14, 16, 12,
        10, 8, 7, 6, 13
    ],

    # Direct nominal protection contribution.
    # Synthetic for mathematical validation only.
    "base_protection": [
        0.05, 0.06, 0.11, 0.10, 0.09,
        0.14, 0.14, 0.11, 0.12, 0.08
    ]
})

assert len(controls) == N_CONTROLS
assert np.isclose(controls["base_protection"].sum(), 1.0)

print("=" * 80)
print("CASCADE — STEP 1.2: SYNTHETIC DEFENSIVE CAPABILITIES")
print("=" * 80)

print(controls.to_string(index=False))

print("\nTotal full resource requirement:",
      controls["resource_cost"].sum())

print("Total nominal direct protection:",
      round(controls["base_protection"].sum(), 4))

print("\n[NOTE] Costs/protection values are synthetic in Step 1.")
print("[NOTE] They are NOT empirical cybersecurity claims.")
print("\n[OK] Step 1.2 completed.")

CASCADE — STEP 1.2: SYNTHETIC DEFENSIVE CAPABILITIES
 id                 name  resource_cost  base_protection
 C1            Telemetry              8             0.05
 C2     Security Logging             10             0.06
 C3   Network Monitoring             14             0.11
 C4        SIEM Analysis             16             0.10
 C5   Automated Response             12             0.09
 C6 Network Segmentation             10             0.14
 C7   Firewall Filtering              8             0.14
 C8        Rate Limiting              7             0.11
 C9       Access Control              6             0.12
C10      Backup Recovery             13             0.08

Total full resource requirement: 104
Total nominal direct protection: 1.0

[NOTE] Costs/protection values are synthetic in Step 1.
[NOTE] They are NOT empirical cybersecurity claims.

[OK] Step 1.2 completed.


Step 1.3 — Construct the Defense Dependency Graph

In [3]:
# ============================================================
# CASCADE — STEP 1.3
# Defense Dependency Graph (DDG)
# ============================================================

# Format:
# upstream_control : [(downstream_control, dependency_strength), ...]

DDG = {
    "C1": [("C3", 0.70), ("C4", 0.65)],
    "C2": [("C4", 0.75)],
    "C3": [("C4", 0.60), ("C5", 0.55)],
    "C4": [("C5", 0.80)],
    "C6": [("C7", 0.35)],
    "C7": [("C8", 0.25)],
    "C9": [("C5", 0.30)]
}

edges = []

for source, targets in DDG.items():
    for target, weight in targets:
        edges.append({
            "source": source,
            "target": target,
            "dependency_strength": weight
        })

edges_df = pd.DataFrame(edges)

print("=" * 80)
print("CASCADE — STEP 1.3: DEFENSE DEPENDENCY GRAPH")
print("=" * 80)

print(edges_df.to_string(index=False))

print(f"\nControls : {N_CONTROLS}")
print(f"Edges    : {len(edges_df)}")

# Check graph direction is acyclic for Step 1.
def has_cycle(graph):
    visited = set()
    active = set()

    def dfs(node):
        if node in active:
            return True
        if node in visited:
            return False

        visited.add(node)
        active.add(node)

        for child, _ in graph.get(node, []):
            if dfs(child):
                return True

        active.remove(node)
        return False

    nodes = controls["id"].tolist()

    return any(dfs(node) for node in nodes if node not in visited)

cycle_found = has_cycle(DDG)

print(f"Cycle detected : {cycle_found}")

assert not cycle_found, \
    "Step 1 requires an acyclic DDG."

print("\n[OK] DDG validated as acyclic.")
print("[OK] Step 1.3 completed.")

CASCADE — STEP 1.3: DEFENSE DEPENDENCY GRAPH
source target  dependency_strength
    C1     C3                 0.70
    C1     C4                 0.65
    C2     C4                 0.75
    C3     C4                 0.60
    C3     C5                 0.55
    C4     C5                 0.80
    C6     C7                 0.35
    C7     C8                 0.25
    C9     C5                 0.30

Controls : 10
Edges    : 9
Cycle detected : False

[OK] DDG validated as acyclic.
[OK] Step 1.3 completed.


Step 1.4 — Implement dependency propagation

In [4]:
# ============================================================
# CASCADE — STEP 1.4
# Dependency propagation
# ============================================================

control_ids = controls["id"].tolist()

# Build incoming dependencies.
incoming = {cid: [] for cid in control_ids}

for source, targets in DDG.items():
    for target, weight in targets:
        incoming[target].append((source, weight))


def propagate_capabilities(configured):
    """
    configured:
        dict {control_id: capability_level}

    Returns:
        effective capability after dependency propagation.

    Assumes DAG.
    """

    effective = {}
    unresolved = set(control_ids)

    while unresolved:

        progress = False

        for cid in list(unresolved):

            parents = incoming[cid]

            # Can calculate when all parents have been calculated.
            if all(parent in effective for parent, _ in parents):

                dep_factor = 1.0

                for parent, weight in parents:

                    parent_eff = effective[parent]

                    factor = 1.0 - weight * (1.0 - parent_eff)

                    dep_factor *= factor

                effective[cid] = configured[cid] * dep_factor

                unresolved.remove(cid)
                progress = True

        if not progress:
            raise RuntimeError(
                "Dependency propagation stalled. "
                "Check DDG for cycles or invalid references."
            )

    return effective


# ------------------------------------------------------------
# TEST A — everything fully operational
# ------------------------------------------------------------

X_full = {cid: 1.0 for cid in control_ids}

EFF_full = propagate_capabilities(X_full)

print("=" * 80)
print("CASCADE — STEP 1.4: DEPENDENCY PROPAGATION")
print("=" * 80)

print("\nTEST A — ALL CONTROLS AT 100%")

for cid in control_ids:
    print(
        f"{cid:>3} configured={X_full[cid]:.2f} "
        f"effective={EFF_full[cid]:.4f}"
    )

assert all(
    np.isclose(EFF_full[cid], 1.0)
    for cid in control_ids
)

print("\n[PASS] Full configuration produces full effective capability.")


# ------------------------------------------------------------
# TEST B — Telemetry degraded
# ------------------------------------------------------------

X_test = X_full.copy()
X_test["C1"] = 0.50

EFF_test = propagate_capabilities(X_test)

print("\nTEST B — C1 TELEMETRY REDUCED TO 50%")

comparison = pd.DataFrame({
    "control": control_ids,
    "configured": [X_test[c] for c in control_ids],
    "effective": [EFF_test[c] for c in control_ids],
})

comparison["dependency_loss"] = (
    comparison["configured"] -
    comparison["effective"]
)

print(comparison.to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

print("\nKey cascade:")

for cid in ["C1", "C3", "C4", "C5"]:
    print(
        f"{cid}: configured={X_test[cid]:.2f}, "
        f"effective={EFF_test[cid]:.4f}"
    )

assert EFF_test["C3"] < X_test["C3"]
assert EFF_test["C4"] < X_test["C4"]
assert EFF_test["C5"] < X_test["C5"]

print("\n[PASS] Upstream degradation propagated downstream.")
print("[OK] Step 1.4 completed.")

CASCADE — STEP 1.4: DEPENDENCY PROPAGATION

TEST A — ALL CONTROLS AT 100%
 C1 configured=1.00 effective=1.0000
 C2 configured=1.00 effective=1.0000
 C3 configured=1.00 effective=1.0000
 C4 configured=1.00 effective=1.0000
 C5 configured=1.00 effective=1.0000
 C6 configured=1.00 effective=1.0000
 C7 configured=1.00 effective=1.0000
 C8 configured=1.00 effective=1.0000
 C9 configured=1.00 effective=1.0000
C10 configured=1.00 effective=1.0000

[PASS] Full configuration produces full effective capability.

TEST B — C1 TELEMETRY REDUCED TO 50%
control  configured  effective  dependency_loss
     C1      0.5000     0.5000           0.0000
     C2      1.0000     1.0000           0.0000
     C3      1.0000     0.6500           0.3500
     C4      1.0000     0.5333           0.4667
     C5      1.0000     0.5060           0.4940
     C6      1.0000     1.0000           0.0000
     C7      1.0000     1.0000           0.0000
     C8      1.0000     1.0000           0.0000
     C9      1.0000    

Step 1.5 — Protection and resource functions

In [5]:
# ============================================================
# CASCADE — STEP 1.5
# System protection and resource consumption
# ============================================================

base_protection = dict(
    zip(controls["id"], controls["base_protection"])
)

resource_cost = dict(
    zip(controls["id"], controls["resource_cost"])
)


def system_protection(configured, propagate=True):
    """
    Weighted defensive protection.

    propagate=True:
        protection uses dependency-adjusted effective capability.

    propagate=False:
        ignores dependencies.
    """

    if propagate:
        capability = propagate_capabilities(configured)
    else:
        capability = configured

    return sum(
        base_protection[cid] * capability[cid]
        for cid in control_ids
    )


def resource_usage(configured):
    """
    Resource usage depends on configured operating level,
    not effective defensive capability.
    """
    return sum(
        resource_cost[cid] * configured[cid]
        for cid in control_ids
    )


print("=" * 80)
print("CASCADE — STEP 1.5: PROTECTION AND RESOURCE MODEL")
print("=" * 80)

# ------------------------------------------------------------
# Full system
# ------------------------------------------------------------

P_full = system_protection(X_full, propagate=True)
R_full = resource_usage(X_full)

print("\nFULL CONFIGURATION")
print(f"Protection     : {P_full:.6f}")
print(f"Resource usage : {R_full:.2f}")

assert np.isclose(P_full, 1.0)

# ------------------------------------------------------------
# C1 degraded to 50%
# ------------------------------------------------------------

P_test_dep = system_protection(X_test, propagate=True)
P_test_nodep = system_protection(X_test, propagate=False)
R_test = resource_usage(X_test)

print("\nC1 = 50%")

print(f"Protection with dependencies    : {P_test_dep:.6f}")
print(f"Protection ignoring dependencies: {P_test_nodep:.6f}")
print(f"Resource usage                  : {R_test:.2f}")

direct_loss = P_full - P_test_nodep
total_loss = P_full - P_test_dep
cascade_component = total_loss - direct_loss

print("\nLOSS DECOMPOSITION")

print(f"Direct loss      : {direct_loss:.6f}")
print(f"Total loss       : {total_loss:.6f}")
print(f"Cascade component: {cascade_component:.6f}")

assert total_loss >= direct_loss
assert cascade_component > 0

print("\n[PASS] Dependency propagation increases total defensive loss.")
print("[OK] Step 1.5 completed.")

CASCADE — STEP 1.5: PROTECTION AND RESOURCE MODEL

FULL CONFIGURATION
Protection     : 1.000000
Resource usage : 104.00

C1 = 50%
Protection with dependencies    : 0.845363
Protection ignoring dependencies: 0.975000
Resource usage                  : 100.00

LOSS DECOMPOSITION
Direct loss      : 0.025000
Total loss       : 0.154637
Cascade component: 0.129637

[PASS] Dependency propagation increases total defensive loss.
[OK] Step 1.5 completed.


Step 1.6 — Formalize Cascading Defensive Loss

In [6]:
# ============================================================
# CASCADE — STEP 1.6
# Cascading Defensive Loss (CDL)
# ============================================================

LEVEL_STEP = 0.25


def degrade_one_level(configured, cid, step=LEVEL_STEP):

    new_config = configured.copy()

    new_config[cid] = max(
        0.0,
        configured[cid] - step
    )

    return new_config


def calculate_cdl(configured, cid, step=LEVEL_STEP):
    """
    Calculate:
        direct loss,
        total dependency-aware loss,
        cascading defensive loss.
    """

    baseline_dep = system_protection(
        configured,
        propagate=True
    )

    degraded = degrade_one_level(
        configured,
        cid,
        step
    )

    # Direct-only counterfactual:
    # same baseline protection model, but only direct capability
    # change is counted.
    direct_loss = (
        base_protection[cid]
        *
        (configured[cid] - degraded[cid])
    )

    degraded_dep = system_protection(
        degraded,
        propagate=True
    )

    total_loss = baseline_dep - degraded_dep

    cdl = total_loss - direct_loss

    return {
        "control": cid,
        "direct_loss": direct_loss,
        "total_loss": total_loss,
        "CDL": cdl
    }


cdl_results = []

for cid in control_ids:
    cdl_results.append(
        calculate_cdl(X_full, cid)
    )

cdl_df = pd.DataFrame(cdl_results)

cdl_df["cascade_multiplier"] = (
    cdl_df["total_loss"]
    /
    cdl_df["direct_loss"]
)

cdl_df = cdl_df.merge(
    controls[["id", "name", "resource_cost"]],
    left_on="control",
    right_on="id"
).drop(columns="id")

cdl_df = cdl_df[
    [
        "control",
        "name",
        "resource_cost",
        "direct_loss",
        "CDL",
        "total_loss",
        "cascade_multiplier"
    ]
]

cdl_df = cdl_df.sort_values(
    "total_loss",
    ascending=False
).reset_index(drop=True)

print("=" * 80)
print("CASCADE — STEP 1.6: CASCADING DEFENSIVE LOSS")
print("=" * 80)

print(
    cdl_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nLargest total-loss control:")
print(
    cdl_df.iloc[0][
        [
            "control",
            "name",
            "direct_loss",
            "CDL",
            "total_loss",
            "cascade_multiplier"
        ]
    ]
)

# Sanity checks
assert (cdl_df["CDL"] >= -1e-10).all()

print("\n[PASS] CDL computed for all controls.")
print("[OK] Step 1.6 completed.")

CASCADE — STEP 1.6: CASCADING DEFENSIVE LOSS
control                 name  resource_cost  direct_loss       CDL  total_loss  cascade_multiplier
     C1            Telemetry              8     0.012500  0.069252    0.081752            6.540177
     C3   Network Monitoring             14     0.027500  0.036690    0.064190            2.334182
     C6 Network Segmentation             10     0.035000  0.014656    0.049656            1.418750
     C2     Security Logging             10     0.015000  0.032250    0.047250            3.150000
     C4        SIEM Analysis             16     0.025000  0.018000    0.043000            1.720000
     C7   Firewall Filtering              8     0.035000  0.006875    0.041875            1.196429
     C9       Access Control              6     0.030000  0.006750    0.036750            1.225000
     C8        Rate Limiting              7     0.027500 -0.000000    0.027500            1.000000
     C5   Automated Response             12     0.022500 -0.0000

Step 1.7 — Marginal Cascading Security Value (MCSV)

In [7]:
# ============================================================
# CASCADE — STEP 1.7
# Marginal Cascading Security Value (MCSV)
# ============================================================

def calculate_mcsv(configured, cid, step=LEVEL_STEP):

    current_level = configured[cid]

    if current_level <= 0:
        return None

    degraded = degrade_one_level(
        configured,
        cid,
        step
    )

    P_before = system_protection(
        configured,
        propagate=True
    )

    P_after = system_protection(
        degraded,
        propagate=True
    )

    R_before = resource_usage(configured)
    R_after = resource_usage(degraded)

    protection_loss = P_before - P_after
    resource_saved = R_before - R_after

    if resource_saved <= 0:
        return None

    mcsv = protection_loss / resource_saved

    # Conventional direct-only value for comparison
    direct_loss = (
        base_protection[cid]
        *
        (configured[cid] - degraded[cid])
    )

    direct_ratio = direct_loss / resource_saved

    return {
        "control": cid,
        "from_level": current_level,
        "to_level": degraded[cid],
        "resource_saved": resource_saved,
        "direct_loss": direct_loss,
        "system_loss": protection_loss,
        "direct_ratio": direct_ratio,
        "MCSV": mcsv
    }


mcsv_results = []

for cid in control_ids:

    result = calculate_mcsv(
        X_full,
        cid
    )

    if result is not None:
        mcsv_results.append(result)


mcsv_df = pd.DataFrame(mcsv_results)

mcsv_df = mcsv_df.merge(
    controls[["id", "name"]],
    left_on="control",
    right_on="id"
).drop(columns="id")

mcsv_df = mcsv_df[
    [
        "control",
        "name",
        "from_level",
        "to_level",
        "resource_saved",
        "direct_loss",
        "system_loss",
        "direct_ratio",
        "MCSV"
    ]
]

mcsv_df = mcsv_df.sort_values(
    "MCSV",
    ascending=True
).reset_index(drop=True)

print("=" * 90)
print("CASCADE — STEP 1.7: MARGINAL CASCADING SECURITY VALUE")
print("=" * 90)

print(
    mcsv_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nPreferred FIRST degradation according to CASCADE:")
print(
    mcsv_df.iloc[0][
        [
            "control",
            "name",
            "resource_saved",
            "system_loss",
            "MCSV"
        ]
    ]
)

print("\nControl that appears cheapest using DIRECT loss only:")

direct_best = (
    mcsv_df
    .sort_values("direct_ratio")
    .iloc[0]
)

print(
    direct_best[
        [
            "control",
            "name",
            "resource_saved",
            "direct_loss",
            "direct_ratio"
        ]
    ]
)

print("\n[OK] Step 1.7 completed.")

CASCADE — STEP 1.7: MARGINAL CASCADING SECURITY VALUE
control                 name  from_level  to_level  resource_saved  direct_loss  system_loss  direct_ratio     MCSV
    C10      Backup Recovery    1.000000  0.750000        3.250000     0.020000     0.020000      0.006154 0.006154
     C5   Automated Response    1.000000  0.750000        3.000000     0.022500     0.022500      0.007500 0.007500
     C4        SIEM Analysis    1.000000  0.750000        4.000000     0.025000     0.043000      0.006250 0.010750
     C8        Rate Limiting    1.000000  0.750000        1.750000     0.027500     0.027500      0.015714 0.015714
     C3   Network Monitoring    1.000000  0.750000        3.500000     0.027500     0.064190      0.007857 0.018340
     C2     Security Logging    1.000000  0.750000        2.500000     0.015000     0.047250      0.006000 0.018900
     C6 Network Segmentation    1.000000  0.750000        2.500000     0.035000     0.049656      0.014000 0.019862
     C7   Firewall

Step 1.8 — First CASCADE degradation experiment

In [8]:
# ============================================================
# CASCADE — STEP 1.8
# Iterative graph-aware degradation
# ============================================================

def cascade_degrade(
    initial_config,
    resource_budget,
    verbose=True
):
    """
    Iteratively degrade one capability level at a time
    using minimum MCSV until resource usage <= budget.
    """

    X = initial_config.copy()

    history = []

    iteration = 0

    while resource_usage(X) > resource_budget:

        iteration += 1

        candidates = []

        for cid in control_ids:

            result = calculate_mcsv(
                X,
                cid
            )

            if result is not None:
                candidates.append(result)

        if not candidates:
            raise RuntimeError(
                "No further degradation possible."
            )

        candidates_df = pd.DataFrame(candidates)

        # Minimum system-wide protection loss
        # per resource unit recovered.
        best = candidates_df.sort_values(
            ["MCSV", "system_loss", "control"],
            ascending=[True, True, True]
        ).iloc[0]

        cid = best["control"]

        before_level = X[cid]

        X = degrade_one_level(
            X,
            cid
        )

        after_level = X[cid]

        history.append({
            "iteration": iteration,
            "control": cid,
            "name": controls.loc[
                controls["id"] == cid,
                "name"
            ].iloc[0],
            "from_level": before_level,
            "to_level": after_level,
            "resource_saved_step":
                best["resource_saved"],
            "system_loss_step":
                best["system_loss"],
            "MCSV":
                best["MCSV"],
            "resource_after":
                resource_usage(X),
            "protection_after":
                system_protection(
                    X,
                    propagate=True
                )
        })

    history_df = pd.DataFrame(history)

    if verbose:

        print("=" * 100)
        print("CASCADE — STEP 1.8: ITERATIVE DEGRADATION")
        print("=" * 100)

        print(
            f"\nInitial resource usage : "
            f"{resource_usage(initial_config):.2f}"
        )

        print(
            f"Resource budget        : "
            f"{resource_budget:.2f}"
        )

        print(
            f"Initial protection     : "
            f"{system_protection(initial_config):.6f}"
        )

        print("\nDECISION TRAJECTORY")

        print(
            history_df.to_string(
                index=False,
                float_format=lambda x: f"{x:.6f}"
            )
        )

        print("\nFINAL CONFIGURATION")

        final_df = pd.DataFrame({
            "control": control_ids,
            "name": [
                controls.loc[
                    controls["id"] == c,
                    "name"
                ].iloc[0]
                for c in control_ids
            ],
            "configured": [
                X[c]
                for c in control_ids
            ],
            "effective": [
                propagate_capabilities(X)[c]
                for c in control_ids
            ]
        })

        print(
            final_df.to_string(
                index=False,
                float_format=lambda x: f"{x:.4f}"
            )
        )

        print(
            f"\nFinal resource usage : "
            f"{resource_usage(X):.2f}"
        )

        print(
            f"Final protection     : "
            f"{system_protection(X):.6f}"
        )

        print(
            f"Protection retained  : "
            f"{100 * system_protection(X) / system_protection(initial_config):.2f}%"
        )

    return X, history_df


RESOURCE_BUDGET = 85.0

X_cascade_85, cascade_history_85 = (
    cascade_degrade(
        X_full,
        RESOURCE_BUDGET
    )
)

CASCADE — STEP 1.8: ITERATIVE DEGRADATION

Initial resource usage : 104.00
Resource budget        : 85.00
Initial protection     : 1.000000

DECISION TRAJECTORY
 iteration control               name  from_level  to_level  resource_saved_step  system_loss_step     MCSV  resource_after  protection_after
         1     C10    Backup Recovery    1.000000  0.750000             3.250000          0.020000 0.006154      100.750000          0.980000
         2     C10    Backup Recovery    0.750000  0.500000             3.250000          0.020000 0.006154       97.500000          0.960000
         3     C10    Backup Recovery    0.500000  0.250000             3.250000          0.020000 0.006154       94.250000          0.940000
         4     C10    Backup Recovery    0.250000  0.000000             3.250000          0.020000 0.006154       91.000000          0.920000
         5      C5 Automated Response    1.000000  0.750000             3.000000          0.022500 0.007500       88.000000      

Step 1.9 — Compare against conventional benefit/cost degradation

In [9]:
# ============================================================
# CASCADE — STEP 1.9
# Dependency-unaware benefit/cost baseline
# ============================================================

def direct_ratio_degrade(
    initial_config,
    resource_budget
):
    """
    Greedy degradation using direct protection/resource ratio.
    Dependencies are ignored for DECISION MAKING,
    but final performance is evaluated using the true
    dependency-aware model.
    """

    X = initial_config.copy()

    history = []

    iteration = 0

    while resource_usage(X) > resource_budget:

        iteration += 1

        candidates = []

        for cid in control_ids:

            if X[cid] <= 0:
                continue

            degraded = degrade_one_level(
                X,
                cid
            )

            resource_saved = (
                resource_usage(X)
                -
                resource_usage(degraded)
            )

            direct_loss = (
                base_protection[cid]
                *
                (X[cid] - degraded[cid])
            )

            ratio = (
                direct_loss / resource_saved
            )

            candidates.append({
                "control": cid,
                "ratio": ratio,
                "resource_saved": resource_saved,
                "direct_loss": direct_loss
            })

        candidates_df = pd.DataFrame(candidates)

        best = candidates_df.sort_values(
            ["ratio", "direct_loss", "control"]
        ).iloc[0]

        cid = best["control"]

        before_level = X[cid]

        X = degrade_one_level(
            X,
            cid
        )

        history.append({
            "iteration": iteration,
            "control": cid,
            "name": controls.loc[
                controls["id"] == cid,
                "name"
            ].iloc[0],
            "from_level": before_level,
            "to_level": X[cid],
            "decision_ratio":
                best["ratio"],
            "resource_after":
                resource_usage(X),
            "true_protection_after":
                system_protection(
                    X,
                    propagate=True
                )
        })

    return X, pd.DataFrame(history)


X_direct_85, direct_history_85 = (
    direct_ratio_degrade(
        X_full,
        RESOURCE_BUDGET
    )
)


print("=" * 90)
print("CASCADE — STEP 1.9: FIRST BASELINE COMPARISON")
print("=" * 90)

comparison_85 = pd.DataFrame({
    "method": [
        "CASCADE",
        "Direct benefit/cost"
    ],

    "resource_usage": [
        resource_usage(X_cascade_85),
        resource_usage(X_direct_85)
    ],

    "true_protection": [
        system_protection(
            X_cascade_85,
            propagate=True
        ),
        system_protection(
            X_direct_85,
            propagate=True
        )
    ]
})

comparison_85["protection_retained_pct"] = (
    100
    *
    comparison_85["true_protection"]
    /
    system_protection(
        X_full,
        propagate=True
    )
)

print(
    comparison_85.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nCASCADE trajectory:")
print(
    cascade_history_85[
        [
            "iteration",
            "control",
            "from_level",
            "to_level",
            "MCSV",
            "resource_after",
            "protection_after"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nDirect benefit/cost trajectory:")
print(
    direct_history_85.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

delta = (
    system_protection(X_cascade_85)
    -
    system_protection(X_direct_85)
)

print(
    f"\nProtection advantage of CASCADE: "
    f"{delta:.6f}"
)

print("\n[OK] Step 1.9 completed.")

CASCADE — STEP 1.9: FIRST BASELINE COMPARISON
             method  resource_usage  true_protection  protection_retained_pct
            CASCADE       85.000000         0.875000                87.500000
Direct benefit/cost       84.250000         0.751000                75.100000

CASCADE trajectory:
 iteration control  from_level  to_level     MCSV  resource_after  protection_after
         1     C10    1.000000  0.750000 0.006154      100.750000          0.980000
         2     C10    0.750000  0.500000 0.006154       97.500000          0.960000
         3     C10    0.500000  0.250000 0.006154       94.250000          0.940000
         4     C10    0.250000  0.000000 0.006154       91.000000          0.920000
         5      C5    1.000000  0.750000 0.007500       88.000000          0.897500
         6      C5    0.750000  0.500000 0.007500       85.000000          0.875000

Direct benefit/cost trajectory:
 iteration control             name  from_level  to_level  decision_ratio  res

Step 1.10 — Exact optimum at budget 85

In [10]:
# ============================================================
# CASCADE — STEP 1.10
# Exact optimum for small synthetic DDG
# ============================================================

from functools import lru_cache

levels = tuple(CAPABILITY_LEVELS.tolist())
costs = controls["resource_cost"].to_numpy(dtype=float)

BEST_PROTECTION = -np.inf
BEST_CONFIG = None
FEASIBLE_EVALUATED = 0

# Maximum resource allowed
BUDGET = RESOURCE_BUDGET


def exact_search(idx, current_levels, current_resource):
    global BEST_PROTECTION
    global BEST_CONFIG
    global FEASIBLE_EVALUATED

    # Resource already exceeds budget -> prune
    if current_resource > BUDGET + 1e-12:
        return

    # Complete configuration
    if idx == N_CONTROLS:

        FEASIBLE_EVALUATED += 1

        X = {
            control_ids[i]: current_levels[i]
            for i in range(N_CONTROLS)
        }

        P = system_protection(
            X,
            propagate=True
        )

        if P > BEST_PROTECTION + 1e-12:
            BEST_PROTECTION = P
            BEST_CONFIG = X.copy()

        return

    cid = control_ids[idx]
    full_cost = resource_cost[cid]

    # Higher levels first: good solutions found earlier.
    for level in reversed(levels):

        new_resource = (
            current_resource
            +
            full_cost * level
        )

        if new_resource <= BUDGET + 1e-12:

            exact_search(
                idx + 1,
                current_levels + [level],
                new_resource
            )


print("=" * 90)
print("CASCADE — STEP 1.10: EXACT OPTIMUM")
print("=" * 90)

start = time.perf_counter()

exact_search(
    idx=0,
    current_levels=[],
    current_resource=0.0
)

elapsed = time.perf_counter() - start

print(f"\nBudget                 : {BUDGET:.2f}")
print(f"Feasible configurations: {FEASIBLE_EVALUATED:,}")
print(f"Exact search time      : {elapsed:.3f} s")
print(f"Optimal protection     : {BEST_PROTECTION:.6f}")
print(
    f"Optimal resource usage : "
    f"{resource_usage(BEST_CONFIG):.2f}"
)

optimal_df = pd.DataFrame({
    "control": control_ids,
    "name": [
        controls.loc[
            controls["id"] == c,
            "name"
        ].iloc[0]
        for c in control_ids
    ],
    "optimal_level": [
        BEST_CONFIG[c]
        for c in control_ids
    ],
    "cascade_level": [
        X_cascade_85[c]
        for c in control_ids
    ]
})

optimal_df["same"] = (
    np.isclose(
        optimal_df["optimal_level"],
        optimal_df["cascade_level"]
    )
)

print("\nOPTIMUM VS CASCADE")
print(
    optimal_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

P_cascade = system_protection(
    X_cascade_85,
    propagate=True
)

optimality_gap = (
    (BEST_PROTECTION - P_cascade)
    / BEST_PROTECTION
    * 100
)

print(
    f"\nCASCADE protection     : "
    f"{P_cascade:.6f}"
)

print(
    f"Exact optimum          : "
    f"{BEST_PROTECTION:.6f}"
)

print(
    f"CASCADE optimality gap : "
    f"{optimality_gap:.4f}%"
)

print("\n[OK] Step 1.10 completed.")

CASCADE — STEP 1.10: EXACT OPTIMUM

Budget                 : 85.00
Feasible configurations: 9,743,835
Exact search time      : 106.006 s
Optimal protection     : 0.875000
Optimal resource usage : 85.00

OPTIMUM VS CASCADE
control                 name  optimal_level  cascade_level  same
     C1            Telemetry           1.00           1.00  True
     C2     Security Logging           1.00           1.00  True
     C3   Network Monitoring           1.00           1.00  True
     C4        SIEM Analysis           1.00           1.00  True
     C5   Automated Response           0.50           0.50  True
     C6 Network Segmentation           1.00           1.00  True
     C7   Firewall Filtering           1.00           1.00  True
     C8        Rate Limiting           1.00           1.00  True
     C9       Access Control           1.00           1.00  True
    C10      Backup Recovery           0.00           0.00  True

CASCADE protection     : 0.875000
Exact optimum          : 0.8

Step 1.11 — Budget sweep

In [12]:
# ============================================================
# CASCADE — STEP 1.10B
# Runtime comparison at budget = 85
# ============================================================

N_RUNTIME_RUNS = 100

times = []

for _ in range(N_RUNTIME_RUNS):

    start = time.perf_counter()

    X_tmp, _ = cascade_degrade(
        X_full,
        RESOURCE_BUDGET,
        verbose=False
    )

    times.append(
        time.perf_counter() - start
    )

times = np.array(times)

print("=" * 80)
print("CASCADE — STEP 1.10B: RUNTIME")
print("=" * 80)

print(f"Runs                 : {N_RUNTIME_RUNS}")
print(f"Mean CASCADE runtime : {times.mean():.6f} s")
print(f"Median runtime       : {np.median(times):.6f} s")
print(f"Std runtime          : {times.std(ddof=1):.6f} s")
print(f"Min runtime          : {times.min():.6f} s")
print(f"Max runtime          : {times.max():.6f} s")

print(f"\nExact search runtime : {elapsed:.3f} s")

speedup = elapsed / times.mean()

print(f"Approx. speedup      : {speedup:,.1f}x")

assert all(
    np.isclose(
        X_tmp[c],
        BEST_CONFIG[c]
    )
    for c in control_ids
)

print("\n[PASS] Repeated CASCADE runs reproduce exact-optimal configuration.")
print("[OK] Step 1.10B completed.")

CASCADE — STEP 1.10B: RUNTIME
Runs                 : 100
Mean CASCADE runtime : 0.012120 s
Median runtime       : 0.011941 s
Std runtime          : 0.000423 s
Min runtime          : 0.011868 s
Max runtime          : 0.014218 s

Exact search runtime : 106.006 s
Approx. speedup      : 8,746.2x

[PASS] Repeated CASCADE runs reproduce exact-optimal configuration.
[OK] Step 1.10B completed.


In [11]:
# ============================================================
# CASCADE — STEP 1.11
# Resource-budget sweep
# ============================================================

budgets = [
    100, 95, 90, 85,
    80, 75, 70, 65,
    60, 55, 50
]

sweep_results = []

for budget in budgets:

    X_cas, _ = cascade_degrade(
        X_full,
        budget,
        verbose=False
    )

    X_dir, _ = direct_ratio_degrade(
        X_full,
        budget
    )

    P_cas = system_protection(
        X_cas,
        propagate=True
    )

    P_dir = system_protection(
        X_dir,
        propagate=True
    )

    sweep_results.append({
        "budget": budget,
        "cascade_resource":
            resource_usage(X_cas),
        "cascade_protection":
            P_cas,
        "direct_resource":
            resource_usage(X_dir),
        "direct_protection":
            P_dir,
        "cascade_advantage":
            P_cas - P_dir
    })


sweep_df = pd.DataFrame(sweep_results)

print("=" * 95)
print("CASCADE — STEP 1.11: RESOURCE-BUDGET SWEEP")
print("=" * 95)

print(
    sweep_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nSUMMARY")

print(
    f"Mean CASCADE advantage : "
    f"{sweep_df['cascade_advantage'].mean():.6f}"
)

print(
    f"Maximum advantage      : "
    f"{sweep_df['cascade_advantage'].max():.6f}"
)

print(
    f"Minimum advantage      : "
    f"{sweep_df['cascade_advantage'].min():.6f}"
)

wins = (
    sweep_df["cascade_advantage"] > 1e-12
).sum()

ties = (
    np.abs(
        sweep_df["cascade_advantage"]
    ) <= 1e-12
).sum()

losses = (
    sweep_df["cascade_advantage"] < -1e-12
).sum()

print(
    f"CASCADE wins/ties/losses: "
    f"{wins}/{ties}/{losses}"
)

print("\n[OK] Step 1.11 completed.")

CASCADE — STEP 1.11: RESOURCE-BUDGET SWEEP
 budget  cascade_resource  cascade_protection  direct_resource  direct_protection  cascade_advantage
    100         97.500000            0.960000        99.000000           0.905500           0.054500
     95         94.250000            0.940000        94.000000           0.811000           0.129000
     90         88.000000            0.897500        87.500000           0.771000           0.126500
     85         85.000000            0.875000        84.250000           0.751000           0.124000
     80         78.000000            0.823000        79.000000           0.685450           0.137550
     75         75.000000            0.805000        75.000000           0.600824           0.204176
     70         67.000000            0.755000        69.000000           0.559562           0.195438
     65         63.000000            0.730000        65.000000           0.557731           0.172269
     60         58.000000            0.700000   

Step 1.12 — Adversarial randomized validation

In [13]:
# ============================================================
# CASCADE — STEP 1.12A
# Randomized adversarial DDG generator
# ============================================================

RNG = np.random.default_rng(20260929)

RAND_N = 8
RAND_LEVELS = np.array([0.00, 0.25, 0.50, 0.75, 1.00])

def generate_random_problem(
    n=RAND_N,
    edge_probability=0.28,
    rng=RNG
):
    """
    Generate a random acyclic defensive dependency problem.

    Important:
    - costs are independently generated
    - protection weights are independently generated
    - dependency strengths are independently generated
    - DAG ordering guarantees no cycles
    """

    ids = [f"R{i+1}" for i in range(n)]

    # Resource costs
    costs = {
        cid: int(rng.integers(5, 21))
        for cid in ids
    }

    # Positive random protection values, normalized to sum to 1
    raw_p = rng.uniform(0.03, 0.20, size=n)
    raw_p = raw_p / raw_p.sum()

    protection = {
        cid: float(raw_p[i])
        for i, cid in enumerate(ids)
    }

    # Random DAG
    graph = {cid: [] for cid in ids}

    for i in range(n):
        for j in range(i + 1, n):

            if rng.random() < edge_probability:

                weight = float(
                    rng.uniform(0.15, 0.85)
                )

                graph[ids[i]].append(
                    (ids[j], weight)
                )

    # Guarantee at least one dependency
    edge_count = sum(
        len(v) for v in graph.values()
    )

    if edge_count == 0:
        graph[ids[0]].append(
            (ids[1], 0.50)
        )

    return {
        "ids": ids,
        "costs": costs,
        "protection": protection,
        "graph": graph
    }


test_problem = generate_random_problem()

print("=" * 85)
print("CASCADE — STEP 1.12A: RANDOM PROBLEM GENERATOR")
print("=" * 85)

print("\nCONTROL PARAMETERS")

test_df = pd.DataFrame({
    "control": test_problem["ids"],
    "resource_cost": [
        test_problem["costs"][c]
        for c in test_problem["ids"]
    ],
    "protection_weight": [
        test_problem["protection"][c]
        for c in test_problem["ids"]
    ]
})

print(
    test_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nDEPENDENCIES")

edge_rows = []

for source, targets in test_problem["graph"].items():
    for target, weight in targets:
        edge_rows.append(
            [source, target, weight]
        )

edge_test_df = pd.DataFrame(
    edge_rows,
    columns=[
        "source",
        "target",
        "weight"
    ]
)

print(
    edge_test_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print(
    f"\nEdges: {len(edge_test_df)}"
)

print("\n[OK] Step 1.12A completed.")

CASCADE — STEP 1.12A: RANDOM PROBLEM GENERATOR

CONTROL PARAMETERS
control  resource_cost  protection_weight
     R1              8           0.037106
     R2             10           0.149779
     R3             13           0.112887
     R4             17           0.174205
     R5             18           0.180400
     R6             13           0.097134
     R7             17           0.103788
     R8              6           0.144701

DEPENDENCIES
source target  weight
    R1     R3  0.7112
    R2     R4  0.2541
    R2     R5  0.6823
    R2     R8  0.4876
    R3     R5  0.3564
    R3     R7  0.3282
    R4     R6  0.4571
    R5     R7  0.2901
    R6     R7  0.8201
    R7     R8  0.4052

Edges: 10

[OK] Step 1.12A completed.


Step 1.12B — Generic propagation/protection engine

In [14]:
# ============================================================
# CASCADE — STEP 1.12B
# Generic randomized-problem engine
# ============================================================

def build_incoming(problem):

    incoming = {
        c: []
        for c in problem["ids"]
    }

    for source, targets in problem["graph"].items():

        for target, weight in targets:
            incoming[target].append(
                (source, weight)
            )

    return incoming


def rand_propagate(problem, X):

    ids = problem["ids"]
    incoming = build_incoming(problem)

    effective = {}
    unresolved = set(ids)

    while unresolved:

        progress = False

        for cid in list(unresolved):

            parents = incoming[cid]

            if all(
                p in effective
                for p, _ in parents
            ):

                factor = 1.0

                for parent, weight in parents:

                    factor *= (
                        1.0
                        -
                        weight
                        *
                        (1.0 - effective[parent])
                    )

                effective[cid] = (
                    X[cid] * factor
                )

                unresolved.remove(cid)
                progress = True

        if not progress:
            raise RuntimeError(
                "Propagation failed."
            )

    return effective


def rand_protection(problem, X):

    eff = rand_propagate(
        problem,
        X
    )

    return sum(
        problem["protection"][c]
        *
        eff[c]
        for c in problem["ids"]
    )


def rand_resource(problem, X):

    return sum(
        problem["costs"][c]
        *
        X[c]
        for c in problem["ids"]
    )


X_rand_full = {
    c: 1.0
    for c in test_problem["ids"]
}

P_rand_full = rand_protection(
    test_problem,
    X_rand_full
)

R_rand_full = rand_resource(
    test_problem,
    X_rand_full
)

print("=" * 85)
print("CASCADE — STEP 1.12B: GENERIC ENGINE")
print("=" * 85)

print(
    f"Full protection : "
    f"{P_rand_full:.6f}"
)

print(
    f"Full resources  : "
    f"{R_rand_full:.2f}"
)

assert np.isclose(
    P_rand_full,
    1.0
)

print("\n[PASS] Random problem full-state protection = 1.")
print("[OK] Step 1.12B completed.")

CASCADE — STEP 1.12B: GENERIC ENGINE
Full protection : 1.000000
Full resources  : 102.00

[PASS] Random problem full-state protection = 1.
[OK] Step 1.12B completed.


Step 1.12C — Generic CASCADE algorithm

In [15]:
# ============================================================
# CASCADE — STEP 1.12C
# Generic CASCADE algorithm
# ============================================================

RAND_STEP = 0.25


def rand_degrade_one(X, cid):

    Y = X.copy()

    Y[cid] = max(
        0.0,
        X[cid] - RAND_STEP
    )

    return Y


def rand_mcsv(problem, X, cid):

    if X[cid] <= 0:
        return None

    Y = rand_degrade_one(
        X,
        cid
    )

    P_before = rand_protection(
        problem,
        X
    )

    P_after = rand_protection(
        problem,
        Y
    )

    R_before = rand_resource(
        problem,
        X
    )

    R_after = rand_resource(
        problem,
        Y
    )

    delta_p = (
        P_before - P_after
    )

    delta_r = (
        R_before - R_after
    )

    if delta_r <= 0:
        return None

    return {
        "control": cid,
        "protection_loss": delta_p,
        "resource_saved": delta_r,
        "MCSV": delta_p / delta_r
    }


def rand_cascade(
    problem,
    budget
):

    X = {
        c: 1.0
        for c in problem["ids"]
    }

    iterations = 0

    while (
        rand_resource(problem, X)
        >
        budget + 1e-12
    ):

        iterations += 1

        candidates = []

        for cid in problem["ids"]:

            result = rand_mcsv(
                problem,
                X,
                cid
            )

            if result is not None:
                candidates.append(result)

        if not candidates:
            raise RuntimeError(
                "No CASCADE move available."
            )

        best = min(
            candidates,
            key=lambda z: (
                z["MCSV"],
                z["protection_loss"],
                z["control"]
            )
        )

        X = rand_degrade_one(
            X,
            best["control"]
        )

    return {
        "config": X,
        "protection":
            rand_protection(problem, X),
        "resource":
            rand_resource(problem, X),
        "iterations":
            iterations
    }


# ------------------------------------------------------------
# Test on the random problem from 1.12A
# ------------------------------------------------------------

test_budget = (
    0.80
    *
    rand_resource(
        test_problem,
        X_rand_full
    )
)

test_cascade = rand_cascade(
    test_problem,
    test_budget
)

print("=" * 85)
print("CASCADE — STEP 1.12C: GENERIC CASCADE")
print("=" * 85)

print(
    f"Budget             : "
    f"{test_budget:.2f}"
)

print(
    f"Final resource     : "
    f"{test_cascade['resource']:.2f}"
)

print(
    f"Protection         : "
    f"{test_cascade['protection']:.6f}"
)

print(
    f"Iterations         : "
    f"{test_cascade['iterations']}"
)

print("\nFINAL CONFIGURATION")

for cid in test_problem["ids"]:
    print(
        f"{cid}: "
        f"{test_cascade['config'][cid]:.2f}"
    )

assert (
    test_cascade["resource"]
    <=
    test_budget + 1e-12
)

print("\n[PASS] Generic CASCADE satisfies resource budget.")
print("[OK] Step 1.12C completed.")

CASCADE — STEP 1.12C: GENERIC CASCADE
Budget             : 81.60
Final resource     : 78.50
Protection         : 0.789015
Iterations         : 6

FINAL CONFIGURATION
R1: 1.00
R2: 1.00
R3: 1.00
R4: 1.00
R5: 1.00
R6: 0.50
R7: 0.00
R8: 1.00

[PASS] Generic CASCADE satisfies resource budget.
[OK] Step 1.12C completed.


Step 1.12D exactly as provided

In [16]:
# ============================================================
# CASCADE — STEP 1.12D
# Exact optimizer for randomized 8-control problems
# ============================================================

def rand_exact_optimum(
    problem,
    budget
):

    ids = problem["ids"]

    best_p = -np.inf
    best_X = None
    best_r = None

    feasible_count = 0

    for levels_tuple in product(
        RAND_LEVELS,
        repeat=len(ids)
    ):

        X = {
            ids[i]: float(levels_tuple[i])
            for i in range(len(ids))
        }

        R = rand_resource(
            problem,
            X
        )

        if R > budget + 1e-12:
            continue

        feasible_count += 1

        P = rand_protection(
            problem,
            X
        )

        if P > best_p + 1e-12:

            best_p = P
            best_X = X.copy()
            best_r = R

    return {
        "config": best_X,
        "protection": best_p,
        "resource": best_r,
        "feasible_count":
            feasible_count
    }


print("=" * 85)
print("CASCADE — STEP 1.12D: EXACT RANDOM OPTIMUM")
print("=" * 85)

start = time.perf_counter()

test_opt = rand_exact_optimum(
    test_problem,
    test_budget
)

test_exact_time = (
    time.perf_counter() - start
)

gap = (
    (
        test_opt["protection"]
        -
        test_cascade["protection"]
    )
    /
    test_opt["protection"]
    *
    100
)

print(
    f"Budget              : "
    f"{test_budget:.2f}"
)

print(
    f"CASCADE protection  : "
    f"{test_cascade['protection']:.6f}"
)

print(
    f"Optimal protection  : "
    f"{test_opt['protection']:.6f}"
)

print(
    f"CASCADE resource    : "
    f"{test_cascade['resource']:.2f}"
)

print(
    f"Optimal resource    : "
    f"{test_opt['resource']:.2f}"
)

print(
    f"Optimality gap      : "
    f"{gap:.4f}%"
)

print(
    f"Feasible states     : "
    f"{test_opt['feasible_count']:,}"
)

print(
    f"Exact runtime       : "
    f"{test_exact_time:.3f} s"
)

print("\nCASCADE vs OPTIMUM CONFIGURATION")

compare_random = pd.DataFrame({
    "control":
        test_problem["ids"],

    "CASCADE": [
        test_cascade["config"][c]
        for c in test_problem["ids"]
    ],

    "OPTIMUM": [
        test_opt["config"][c]
        for c in test_problem["ids"]
    ]
})

compare_random["same"] = np.isclose(
    compare_random["CASCADE"],
    compare_random["OPTIMUM"]
)

print(
    compare_random.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)

print("\n[OK] Step 1.12D completed.")

CASCADE — STEP 1.12D: EXACT RANDOM OPTIMUM
Budget              : 81.60
CASCADE protection  : 0.789015
Optimal protection  : 0.798137
CASCADE resource    : 78.50
Optimal resource    : 81.50
Optimality gap      : 1.1429%
Feasible states     : 386,853
Exact runtime       : 6.308 s

CASCADE vs OPTIMUM CONFIGURATION
control  CASCADE  OPTIMUM  same
     R1     1.00     1.00  True
     R2     1.00     1.00  True
     R3     1.00     1.00  True
     R4     1.00     1.00  True
     R5     1.00     0.75 False
     R6     0.50     0.75 False
     R7     0.00     0.25 False
     R8     1.00     1.00  True

[OK] Step 1.12D completed.


Step 1.12E — Randomized multi-graph benchmark

In [17]:
# ============================================================
# CASCADE — STEP 1.12E
# Frozen CASCADE V1 randomized benchmark
#
# 30 random DDGs × 3 resource conditions
# CASCADE V1 vs exact global optimum
# ============================================================

BENCHMARK_SEED = 20260930
N_GRAPHS = 30

RESOURCE_RATIOS = [
    0.80,
    0.65,
    0.50
]

benchmark_rng = np.random.default_rng(
    BENCHMARK_SEED
)

benchmark_results = []
benchmark_problems = {}

print("=" * 100)
print("CASCADE — STEP 1.12E: RANDOMIZED MULTI-GRAPH BENCHMARK")
print("=" * 100)

print(f"Benchmark seed       : {BENCHMARK_SEED}")
print(f"Random DDGs          : {N_GRAPHS}")
print(f"Controls per DDG     : {RAND_N}")
print(f"Capability levels    : {RAND_LEVELS.tolist()}")
print(f"Resource conditions  : {RESOURCE_RATIOS}")
print(
    f"Total comparisons    : "
    f"{N_GRAPHS * len(RESOURCE_RATIOS)}"
)

benchmark_start = time.perf_counter()


# ------------------------------------------------------------
# Generate each graph once, then test all three budgets
# ------------------------------------------------------------

for graph_idx in range(1, N_GRAPHS + 1):

    problem = generate_random_problem(
        n=RAND_N,
        edge_probability=0.28,
        rng=benchmark_rng
    )

    graph_id = f"G{graph_idx:02d}"

    benchmark_problems[graph_id] = problem

    X_full_random = {
        c: 1.0
        for c in problem["ids"]
    }

    full_resource = rand_resource(
        problem,
        X_full_random
    )

    edge_count = sum(
        len(v)
        for v in problem["graph"].values()
    )

    print(
        f"\n[{graph_id}] "
        f"edges={edge_count}, "
        f"full_resource={full_resource:.2f}"
    )

    for ratio in RESOURCE_RATIOS:

        budget = (
            full_resource * ratio
        )

        # ----------------------------------------------------
        # CASCADE V1
        # ----------------------------------------------------

        cascade_start = time.perf_counter()

        cas = rand_cascade(
            problem,
            budget
        )

        cascade_runtime = (
            time.perf_counter()
            -
            cascade_start
        )

        # ----------------------------------------------------
        # Exact global optimum
        # ----------------------------------------------------

        exact_start = time.perf_counter()

        opt = rand_exact_optimum(
            problem,
            budget
        )

        exact_runtime = (
            time.perf_counter()
            -
            exact_start
        )

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        gap_pct = (
            (
                opt["protection"]
                -
                cas["protection"]
            )
            /
            opt["protection"]
            *
            100
        )

        cascade_unused = (
            budget
            -
            cas["resource"]
        )

        optimum_unused = (
            budget
            -
            opt["resource"]
        )

        exact_match = (
            abs(
                cas["protection"]
                -
                opt["protection"]
            )
            <= 1e-10
        )

        result = {
            "graph_id":
                graph_id,

            "resource_ratio":
                ratio,

            "edge_count":
                edge_count,

            "full_resource":
                full_resource,

            "budget":
                budget,

            "cascade_resource":
                cas["resource"],

            "opt_resource":
                opt["resource"],

            "cascade_unused":
                cascade_unused,

            "opt_unused":
                optimum_unused,

            "cascade_protection":
                cas["protection"],

            "opt_protection":
                opt["protection"],

            "gap_pct":
                gap_pct,

            "exact_match":
                exact_match,

            "cascade_iterations":
                cas["iterations"],

            "cascade_runtime_s":
                cascade_runtime,

            "exact_runtime_s":
                exact_runtime,

            "feasible_states":
                opt["feasible_count"]
        }

        benchmark_results.append(
            result
        )

        print(
            f"   Budget={ratio:.0%} | "
            f"CAS={cas['protection']:.6f} | "
            f"OPT={opt['protection']:.6f} | "
            f"Gap={gap_pct:.3f}% | "
            f"Unused={cascade_unused:.2f} | "
            f"ExactMatch={exact_match}"
        )


# ------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------

benchmark_df = pd.DataFrame(
    benchmark_results
)

total_benchmark_time = (
    time.perf_counter()
    -
    benchmark_start
)


print("\n" + "=" * 100)
print("BENCHMARK COMPLETE")
print("=" * 100)

print(
    f"Total runtime : "
    f"{total_benchmark_time:.2f} s "
    f"({total_benchmark_time / 60:.2f} min)"
)

print(
    f"Comparisons   : "
    f"{len(benchmark_df)}"
)

print("\n[OK] Step 1.12E completed.")

CASCADE — STEP 1.12E: RANDOMIZED MULTI-GRAPH BENCHMARK
Benchmark seed       : 20260930
Random DDGs          : 30
Controls per DDG     : 8
Capability levels    : [0.0, 0.25, 0.5, 0.75, 1.0]
Resource conditions  : [0.8, 0.65, 0.5]
Total comparisons    : 90

[G01] edges=6, full_resource=100.00
   Budget=80% | CAS=0.892794 | OPT=0.892794 | Gap=0.000% | Unused=0.00 | ExactMatch=True
   Budget=65% | CAS=0.717323 | OPT=0.717323 | Gap=0.000% | Unused=1.00 | ExactMatch=True
   Budget=50% | CAS=0.554197 | OPT=0.576209 | Gap=3.820% | Unused=2.50 | ExactMatch=False

[G02] edges=8, full_resource=91.00
   Budget=80% | CAS=0.860963 | OPT=0.860963 | Gap=0.000% | Unused=1.80 | ExactMatch=True
   Budget=65% | CAS=0.712541 | OPT=0.715365 | Gap=0.395% | Unused=1.15 | ExactMatch=False
   Budget=50% | CAS=0.519449 | OPT=0.561546 | Gap=7.497% | Unused=3.50 | ExactMatch=False

[G03] edges=6, full_resource=110.00
   Budget=80% | CAS=0.822869 | OPT=0.830545 | Gap=0.924% | Unused=2.00 | ExactMatch=False
   Budge

Step 1.12F — Randomized Benchmark Statistical Summary

In [18]:
# ============================================================
# CASCADE — STEP 1.12F
# Randomized benchmark statistical summary
#
# IMPORTANT:
# Run directly after Step 1.12E.
# Uses the existing benchmark_df.
# ============================================================

print("=" * 100)
print("CASCADE — STEP 1.12F: RANDOMIZED BENCHMARK SUMMARY")
print("=" * 100)

# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert "benchmark_df" in globals(), (
    "benchmark_df not found. "
    "Run Step 1.12E first."
)

assert len(benchmark_df) == 90, (
    f"Expected 90 comparisons, found {len(benchmark_df)}."
)

required_columns = [
    "graph_id",
    "resource_ratio",
    "edge_count",
    "budget",
    "cascade_resource",
    "opt_resource",
    "cascade_unused",
    "opt_unused",
    "cascade_protection",
    "opt_protection",
    "gap_pct",
    "exact_match",
    "cascade_iterations",
    "cascade_runtime_s",
    "exact_runtime_s",
    "feasible_states"
]

missing_columns = [
    c for c in required_columns
    if c not in benchmark_df.columns
]

assert len(missing_columns) == 0, (
    f"Missing columns: {missing_columns}"
)

print(
    f"\nVerified comparisons : "
    f"{len(benchmark_df)}"
)


# ------------------------------------------------------------
# 1. Overall optimality-gap statistics
# ------------------------------------------------------------

gap_values = benchmark_df[
    "gap_pct"
].to_numpy(dtype=float)

exact_matches = int(
    benchmark_df[
        "exact_match"
    ].sum()
)

exact_match_rate = (
    100.0
    *
    exact_matches
    /
    len(benchmark_df)
)

print("\n" + "-" * 100)
print("1. OVERALL OPTIMALITY GAP")
print("-" * 100)

print(
    f"Mean gap        : "
    f"{np.mean(gap_values):.4f}%"
)

print(
    f"Median gap      : "
    f"{np.median(gap_values):.4f}%"
)

print(
    f"Std gap         : "
    f"{np.std(gap_values, ddof=1):.4f}%"
)

print(
    f"Minimum gap     : "
    f"{np.min(gap_values):.4f}%"
)

print(
    f"25th percentile : "
    f"{np.percentile(gap_values, 25):.4f}%"
)

print(
    f"75th percentile : "
    f"{np.percentile(gap_values, 75):.4f}%"
)

print(
    f"90th percentile : "
    f"{np.percentile(gap_values, 90):.4f}%"
)

print(
    f"95th percentile : "
    f"{np.percentile(gap_values, 95):.4f}%"
)

print(
    f"Maximum gap     : "
    f"{np.max(gap_values):.4f}%"
)

print(
    f"\nExact matches   : "
    f"{exact_matches}/{len(benchmark_df)}"
)

print(
    f"Exact-match rate: "
    f"{exact_match_rate:.2f}%"
)


# ------------------------------------------------------------
# 2. Near-optimality thresholds
# ------------------------------------------------------------

print("\n" + "-" * 100)
print("2. NEAR-OPTIMALITY THRESHOLDS")
print("-" * 100)

thresholds = [
    0.0,
    1.0,
    2.0,
    5.0,
    10.0
]

for threshold in thresholds:

    if threshold == 0.0:

        count = int(
            np.sum(
                np.isclose(
                    gap_values,
                    0.0,
                    atol=1e-10
                )
            )
        )

        label = "Exact optimum"

    else:

        count = int(
            np.sum(
                gap_values <= threshold
            )
        )

        label = (
            f"Gap <= {threshold:.0f}%"
        )

    pct = (
        100.0
        *
        count
        /
        len(gap_values)
    )

    print(
        f"{label:<18}: "
        f"{count:>2}/{len(gap_values)} "
        f"({pct:6.2f}%)"
    )


# ------------------------------------------------------------
# 3. Results by resource condition
# ------------------------------------------------------------

summary_by_budget = (
    benchmark_df
    .groupby(
        "resource_ratio"
    )
    .agg(
        comparisons=(
            "gap_pct",
            "size"
        ),

        mean_gap_pct=(
            "gap_pct",
            "mean"
        ),

        median_gap_pct=(
            "gap_pct",
            "median"
        ),

        std_gap_pct=(
            "gap_pct",
            "std"
        ),

        max_gap_pct=(
            "gap_pct",
            "max"
        ),

        mean_cascade_protection=(
            "cascade_protection",
            "mean"
        ),

        mean_opt_protection=(
            "opt_protection",
            "mean"
        ),

        mean_cascade_unused=(
            "cascade_unused",
            "mean"
        ),

        mean_opt_unused=(
            "opt_unused",
            "mean"
        ),

        exact_matches=(
            "exact_match",
            "sum"
        )
    )
    .reset_index()
)

summary_by_budget[
    "exact_match_pct"
] = (
    100.0
    *
    summary_by_budget[
        "exact_matches"
    ]
    /
    summary_by_budget[
        "comparisons"
    ]
)

print("\n" + "-" * 100)
print("3. RESULTS BY RESOURCE CONDITION")
print("-" * 100)

print(
    summary_by_budget.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# 4. Runtime comparison
# ------------------------------------------------------------

mean_cascade_runtime = (
    benchmark_df[
        "cascade_runtime_s"
    ].mean()
)

median_cascade_runtime = (
    benchmark_df[
        "cascade_runtime_s"
    ].median()
)

mean_exact_runtime = (
    benchmark_df[
        "exact_runtime_s"
    ].mean()
)

median_exact_runtime = (
    benchmark_df[
        "exact_runtime_s"
    ].median()
)

runtime_ratio = (
    mean_exact_runtime
    /
    mean_cascade_runtime
)

print("\n" + "-" * 100)
print("4. RUNTIME")
print("-" * 100)

print(
    f"Mean CASCADE runtime   : "
    f"{mean_cascade_runtime:.6f} s"
)

print(
    f"Median CASCADE runtime : "
    f"{median_cascade_runtime:.6f} s"
)

print(
    f"Mean exact runtime     : "
    f"{mean_exact_runtime:.6f} s"
)

print(
    f"Median exact runtime   : "
    f"{median_exact_runtime:.6f} s"
)

print(
    f"Mean runtime ratio     : "
    f"{runtime_ratio:,.1f}x"
)


# ------------------------------------------------------------
# 5. Resource-utilization behavior
# ------------------------------------------------------------

mean_cascade_unused = (
    benchmark_df[
        "cascade_unused"
    ].mean()
)

median_cascade_unused = (
    benchmark_df[
        "cascade_unused"
    ].median()
)

max_cascade_unused = (
    benchmark_df[
        "cascade_unused"
    ].max()
)

mean_opt_unused = (
    benchmark_df[
        "opt_unused"
    ].mean()
)

print("\n" + "-" * 100)
print("5. RESOURCE UTILIZATION")
print("-" * 100)

print(
    f"Mean CASCADE unused resource : "
    f"{mean_cascade_unused:.4f}"
)

print(
    f"Median CASCADE unused        : "
    f"{median_cascade_unused:.4f}"
)

print(
    f"Maximum CASCADE unused       : "
    f"{max_cascade_unused:.4f}"
)

print(
    f"Mean OPT unused resource     : "
    f"{mean_opt_unused:.4f}"
)


# ------------------------------------------------------------
# 6. Relationship between unused resource and gap
# ------------------------------------------------------------

if (
    benchmark_df[
        "cascade_unused"
    ].std() > 0
    and
    benchmark_df[
        "gap_pct"
    ].std() > 0
):

    unused_gap_corr = (
        benchmark_df[
            [
                "cascade_unused",
                "gap_pct"
            ]
        ]
        .corr()
        .iloc[0, 1]
    )

else:

    unused_gap_corr = np.nan


print("\n" + "-" * 100)
print("6. UNUSED RESOURCE vs OPTIMALITY GAP")
print("-" * 100)

print(
    f"Pearson correlation : "
    f"{unused_gap_corr:.4f}"
)

print(
    "[NOTE] This is descriptive only; "
    "it does not establish causality."
)


# ------------------------------------------------------------
# 7. Worst observed case
# ------------------------------------------------------------

worst_idx = (
    benchmark_df[
        "gap_pct"
    ].idxmax()
)

worst_case = (
    benchmark_df.loc[
        worst_idx
    ]
)

print("\n" + "-" * 100)
print("7. WORST OBSERVED CASE")
print("-" * 100)

print(
    worst_case.to_string()
)


# ------------------------------------------------------------
# 8. Top 10 optimality gaps
# ------------------------------------------------------------

top10 = (
    benchmark_df
    .sort_values(
        "gap_pct",
        ascending=False
    )
    .head(10)
)

print("\n" + "-" * 100)
print("8. TOP 10 OPTIMALITY GAPS")
print("-" * 100)

print(
    top10[
        [
            "graph_id",
            "resource_ratio",
            "edge_count",
            "budget",
            "cascade_resource",
            "opt_resource",
            "cascade_protection",
            "opt_protection",
            "gap_pct",
            "cascade_unused",
            "opt_unused",
            "cascade_runtime_s",
            "exact_runtime_s"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ------------------------------------------------------------
# 9. Best/exact cases count by resource condition
# ------------------------------------------------------------

exact_by_budget = (
    benchmark_df
    .groupby(
        "resource_ratio"
    )[
        "exact_match"
    ]
    .agg(
        [
            "sum",
            "count"
        ]
    )
    .reset_index()
)

exact_by_budget[
    "exact_match_pct"
] = (
    100.0
    *
    exact_by_budget["sum"]
    /
    exact_by_budget["count"]
)

print("\n" + "-" * 100)
print("9. EXACT MATCHES BY RESOURCE CONDITION")
print("-" * 100)

print(
    exact_by_budget.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ------------------------------------------------------------
# Final integrity checks
# ------------------------------------------------------------

assert (
    benchmark_df[
        "gap_pct"
    ] >= -1e-8
).all(), (
    "Negative optimality gap detected. "
    "Check exact optimizer."
)

assert (
    benchmark_df[
        "cascade_resource"
    ]
    <=
    benchmark_df[
        "budget"
    ] + 1e-8
).all(), (
    "CASCADE resource-budget violation detected."
)

assert (
    benchmark_df[
        "opt_resource"
    ]
    <=
    benchmark_df[
        "budget"
    ] + 1e-8
).all(), (
    "Exact optimizer resource-budget violation detected."
)

print("\n" + "=" * 100)
print("[PASS] All benchmark integrity checks passed.")
print("[OK] Step 1.12F completed.")
print("=" * 100)

CASCADE — STEP 1.12F: RANDOMIZED BENCHMARK SUMMARY

Verified comparisons : 90

----------------------------------------------------------------------------------------------------
1. OVERALL OPTIMALITY GAP
----------------------------------------------------------------------------------------------------
Mean gap        : 1.2839%
Median gap      : 0.3845%
Std gap         : 2.5903%
Minimum gap     : 0.0000%
25th percentile : 0.0000%
75th percentile : 1.7632%
90th percentile : 3.3333%
95th percentile : 3.8762%
Maximum gap     : 21.6529%

Exact matches   : 36/90
Exact-match rate: 40.00%

----------------------------------------------------------------------------------------------------
2. NEAR-OPTIMALITY THRESHOLDS
----------------------------------------------------------------------------------------------------
Exact optimum     : 36/90 ( 40.00%)
Gap <= 1%         : 56/90 ( 62.22%)
Gap <= 2%         : 71/90 ( 78.89%)
Gap <= 5%         : 88/90 ( 97.78%)
Gap <= 10%        : 89/90 ( 98.

Step 1.12G — Diagnose the G19/50% failure

In [22]:
# ============================================================
# CASCADE — STEP 1.12G (FINAL CORRECTED VERSION)
# Worst-case diagnostic: G19 at 50% resource
#
# Frozen structure confirmed:
# G19 = {
#     "ids": ...,
#     "costs": ...,
#     "protection": ...,
#     "graph": ...
# }
#
# IMPORTANT:
# - Does NOT modify CASCADE V1
# - Does NOT regenerate G19
# - Uses frozen benchmark_problems["G19"]
# ============================================================

print("=" * 100)
print("CASCADE — STEP 1.12G: G19 WORST-CASE DIAGNOSTIC")
print("=" * 100)


# ============================================================
# 1. Recover frozen G19
# ============================================================

assert "benchmark_problems" in globals(), (
    "benchmark_problems not found. "
    "Step 1.12E data must still be in memory."
)

assert "G19" in benchmark_problems, (
    "Frozen G19 not found."
)

G19 = benchmark_problems["G19"]

g19_ids = list(G19["ids"])
g19_costs = G19["costs"]
g19_protection = G19["protection"]
g19_graph = G19["graph"]

print("\nFrozen G19 successfully recovered.")

print(
    f"Controls : {g19_ids}"
)

print(
    f"G19 keys : {list(G19.keys())}"
)


# ============================================================
# 2. Integrity check of stored structure
# ============================================================

for c in g19_ids:

    assert c in g19_costs, (
        f"Missing cost for {c}"
    )

    assert c in g19_protection, (
        f"Missing protection for {c}"
    )

    assert c in g19_graph, (
        f"Missing graph entry for {c}"
    )

print(
    "[PASS] G19 structure verified."
)


# ============================================================
# 3. Resource condition
# ============================================================

X_full_g19 = {
    c: 1.0
    for c in g19_ids
}

g19_full_resource = rand_resource(
    G19,
    X_full_g19
)

g19_budget = (
    0.50
    *
    g19_full_resource
)

print("\n" + "-" * 100)
print("RESOURCE CONDITION")
print("-" * 100)

print(
    f"Full resource : "
    f"{g19_full_resource:.2f}"
)

print(
    f"50% budget    : "
    f"{g19_budget:.2f}"
)


# ============================================================
# 4. Reproduce frozen CASCADE V1 and exact optimum
# ============================================================

g19_cascade = rand_cascade(
    G19,
    g19_budget
)

g19_opt = rand_exact_optimum(
    G19,
    g19_budget
)

g19_gap = (
    (
        g19_opt["protection"]
        -
        g19_cascade["protection"]
    )
    /
    g19_opt["protection"]
    *
    100.0
)

cascade_unused = (
    g19_budget
    -
    g19_cascade["resource"]
)

opt_unused = (
    g19_budget
    -
    g19_opt["resource"]
)


print("\n" + "-" * 100)
print("CASCADE V1 vs EXACT OPTIMUM")
print("-" * 100)

print(
    f"CASCADE protection : "
    f"{g19_cascade['protection']:.6f}"
)

print(
    f"OPT protection     : "
    f"{g19_opt['protection']:.6f}"
)

print(
    f"CASCADE resource   : "
    f"{g19_cascade['resource']:.2f}"
)

print(
    f"OPT resource       : "
    f"{g19_opt['resource']:.2f}"
)

print(
    f"CASCADE unused     : "
    f"{cascade_unused:.2f}"
)

print(
    f"OPT unused         : "
    f"{opt_unused:.2f}"
)

print(
    f"Optimality gap     : "
    f"{g19_gap:.4f}%"
)


# ============================================================
# 5. Control parameters
# ============================================================

control_rows = []

for c in g19_ids:

    control_rows.append({
        "control":
            c,

        "cost":
            float(
                g19_costs[c]
            ),

        "base_protection":
            float(
                g19_protection[c]
            )
    })

g19_control_df = pd.DataFrame(
    control_rows
)

print("\n" + "-" * 100)
print("G19 CONTROL PARAMETERS")
print("-" * 100)

print(
    g19_control_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 6. Dependency edges
# ============================================================

dependency_rows = []

for parent in g19_ids:

    children = g19_graph[parent]

    for item in children:

        child = item[0]
        weight = float(
            item[1]
        )

        dependency_rows.append({
            "parent":
                parent,

            "child":
                child,

            "weight":
                weight
        })

g19_dependency_df = pd.DataFrame(
    dependency_rows
)

print("\n" + "-" * 100)
print("G19 DEPENDENCY EDGES")
print("-" * 100)

if len(
    g19_dependency_df
) > 0:

    print(
        g19_dependency_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )

else:

    print(
        "No dependency edges."
    )

print(
    f"\nNumber of dependency edges : "
    f"{len(g19_dependency_df)}"
)


# ============================================================
# 7. CASCADE vs OPT configured states
# ============================================================

config_rows = []

for c in g19_ids:

    cas_level = float(
        g19_cascade[
            "config"
        ][c]
    )

    opt_level = float(
        g19_opt[
            "config"
        ][c]
    )

    config_rows.append({
        "control":
            c,

        "cost":
            float(
                g19_costs[c]
            ),

        "base_protection":
            float(
                g19_protection[c]
            ),

        "CASCADE_level":
            cas_level,

        "OPT_level":
            opt_level,

        "CAS_minus_OPT":
            (
                cas_level
                -
                opt_level
            )
    })

g19_config_df = pd.DataFrame(
    config_rows
)

print("\n" + "-" * 100)
print("CONFIGURED STATE COMPARISON")
print("-" * 100)

print(
    g19_config_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 8. Effective states after dependency propagation
# ============================================================

cascade_effective = rand_propagate(
    G19,
    g19_cascade["config"]
)

opt_effective = rand_propagate(
    G19,
    g19_opt["config"]
)

effective_rows = []

for c in g19_ids:

    effective_rows.append({
        "control":
            c,

        "CASCADE_config":
            float(
                g19_cascade[
                    "config"
                ][c]
            ),

        "CASCADE_effective":
            float(
                cascade_effective[c]
            ),

        "OPT_config":
            float(
                g19_opt[
                    "config"
                ][c]
            ),

        "OPT_effective":
            float(
                opt_effective[c]
            ),

        "CAS_minus_OPT_effective":
            float(
                cascade_effective[c]
                -
                opt_effective[c]
            )
    })

g19_effective_df = pd.DataFrame(
    effective_rows
)

print("\n" + "-" * 100)
print("EFFECTIVE CAPABILITY COMPARISON")
print("-" * 100)

print(
    g19_effective_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 9. Protection contribution by control
# ============================================================

contribution_rows = []

for c in g19_ids:

    base_p = float(
        g19_protection[c]
    )

    cas_contribution = (
        base_p
        *
        float(
            cascade_effective[c]
        )
    )

    opt_contribution = (
        base_p
        *
        float(
            opt_effective[c]
        )
    )

    contribution_rows.append({
        "control":
            c,

        "base_protection":
            base_p,

        "CASCADE_contribution":
            cas_contribution,

        "OPT_contribution":
            opt_contribution,

        "CAS_minus_OPT":
            (
                cas_contribution
                -
                opt_contribution
            )
    })

g19_contribution_df = pd.DataFrame(
    contribution_rows
)

print("\n" + "-" * 100)
print("PROTECTION CONTRIBUTION BY CONTROL")
print("-" * 100)

print(
    g19_contribution_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)

print("\nContribution totals:")

print(
    f"CASCADE : "
    f"{g19_contribution_df['CASCADE_contribution'].sum():.6f}"
)

print(
    f"OPT     : "
    f"{g19_contribution_df['OPT_contribution'].sum():.6f}"
)


# ============================================================
# 10. Dependency centrality
# ============================================================

incoming_count = {
    c: 0
    for c in g19_ids
}

outgoing_count = {
    c: 0
    for c in g19_ids
}

incoming_weight = {
    c: 0.0
    for c in g19_ids
}

outgoing_weight = {
    c: 0.0
    for c in g19_ids
}

for row in dependency_rows:

    parent = row[
        "parent"
    ]

    child = row[
        "child"
    ]

    weight = float(
        row[
            "weight"
        ]
    )

    outgoing_count[
        parent
    ] += 1

    incoming_count[
        child
    ] += 1

    outgoing_weight[
        parent
    ] += weight

    incoming_weight[
        child
    ] += weight


dependency_summary_rows = []

for c in g19_ids:

    dependency_summary_rows.append({
        "control":
            c,

        "incoming_edges":
            incoming_count[c],

        "outgoing_edges":
            outgoing_count[c],

        "incoming_weight":
            incoming_weight[c],

        "outgoing_weight":
            outgoing_weight[c]
    })

g19_dependency_summary = pd.DataFrame(
    dependency_summary_rows
)

print("\n" + "-" * 100)
print("DEPENDENCY CENTRALITY SUMMARY")
print("-" * 100)

print(
    g19_dependency_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 11. CASCADE degradation trajectory
# ============================================================

print("\n" + "-" * 100)
print("CASCADE V1 DEGRADATION TRAJECTORY")
print("-" * 100)

trajectory = g19_cascade.get(
    "trajectory",
    []
)

if len(
    trajectory
) == 0:

    print(
        "No trajectory stored by rand_cascade()."
    )

else:

    g19_trajectory_df = pd.DataFrame(
        trajectory
    )

    print(
        g19_trajectory_df.to_string(
            index=False
        )
    )


# ============================================================
# 12. Prepare level utilities
# ============================================================

level_array = np.asarray(
    RAND_LEVELS,
    dtype=float
)

current_X = (
    g19_cascade[
        "config"
    ].copy()
)

current_P = rand_protection(
    G19,
    current_X
)

current_R = rand_resource(
    G19,
    current_X
)


def level_index(value):

    return int(
        np.argmin(
            np.abs(
                level_array
                -
                float(value)
            )
        )
    )


# ============================================================
# 13. One-step feasible upgrades
# ============================================================

repair_rows = []

for c in g19_ids:

    current_level = float(
        current_X[c]
    )

    idx = level_index(
        current_level
    )

    if (
        idx
        >=
        len(level_array) - 1
    ):
        continue

    next_level = float(
        level_array[
            idx + 1
        ]
    )

    X_test = (
        current_X.copy()
    )

    X_test[c] = (
        next_level
    )

    R_test = rand_resource(
        G19,
        X_test
    )

    if (
        R_test
        >
        g19_budget + 1e-12
    ):
        continue

    P_test = rand_protection(
        G19,
        X_test
    )

    repair_rows.append({
        "control":
            c,

        "from_level":
            current_level,

        "to_level":
            next_level,

        "resource_before":
            current_R,

        "resource_after":
            R_test,

        "resource_added":
            (
                R_test
                -
                current_R
            ),

        "protection_before":
            current_P,

        "protection_after":
            P_test,

        "protection_gain":
            (
                P_test
                -
                current_P
            )
    })


print("\n" + "-" * 100)
print("ONE-STEP FEASIBLE UPGRADES FROM CASCADE SOLUTION")
print("-" * 100)

if len(
    repair_rows
) == 0:

    g19_repair_df = pd.DataFrame()

    print(
        "No single one-level upgrade fits "
        "inside the remaining budget."
    )

else:

    g19_repair_df = (
        pd.DataFrame(
            repair_rows
        )
        .sort_values(
            "protection_gain",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    print(
        g19_repair_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


# ============================================================
# 14. One-for-one local exchange diagnostic
#
# Upgrade one control by one level and degrade another
# control by one level.
#
# This directly tests GREEDY PATH DEPENDENCE.
# ============================================================

exchange_rows = []

for up_control in g19_ids:

    up_level = float(
        current_X[
            up_control
        ]
    )

    up_idx = level_index(
        up_level
    )

    if (
        up_idx
        >=
        len(level_array) - 1
    ):
        continue

    new_up_level = float(
        level_array[
            up_idx + 1
        ]
    )


    for down_control in g19_ids:

        if (
            down_control
            ==
            up_control
        ):
            continue

        down_level = float(
            current_X[
                down_control
            ]
        )

        down_idx = level_index(
            down_level
        )

        if down_idx <= 0:
            continue

        new_down_level = float(
            level_array[
                down_idx - 1
            ]
        )

        X_exchange = (
            current_X.copy()
        )

        X_exchange[
            up_control
        ] = (
            new_up_level
        )

        X_exchange[
            down_control
        ] = (
            new_down_level
        )

        R_exchange = rand_resource(
            G19,
            X_exchange
        )

        if (
            R_exchange
            >
            g19_budget + 1e-12
        ):
            continue

        P_exchange = rand_protection(
            G19,
            X_exchange
        )

        exchange_rows.append({
            "upgrade":
                up_control,

            "upgrade_from":
                up_level,

            "upgrade_to":
                new_up_level,

            "degrade":
                down_control,

            "degrade_from":
                down_level,

            "degrade_to":
                new_down_level,

            "resource":
                R_exchange,

            "protection":
                P_exchange,

            "gain_vs_CASCADE":
                (
                    P_exchange
                    -
                    current_P
                )
        })


print("\n" + "-" * 100)
print("TOP ONE-FOR-ONE LOCAL EXCHANGES")
print("-" * 100)

if len(
    exchange_rows
) == 0:

    g19_exchange_df = pd.DataFrame()

    print(
        "No feasible one-for-one exchanges found."
    )

else:

    g19_exchange_df = (
        pd.DataFrame(
            exchange_rows
        )
        .sort_values(
            "gain_vs_CASCADE",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    print(
        g19_exchange_df
        .head(15)
        .to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


# ============================================================
# 15. Compare which controls differ from OPT
# ============================================================

difference_rows = []

for c in g19_ids:

    cas_level = float(
        current_X[c]
    )

    opt_level = float(
        g19_opt[
            "config"
        ][c]
    )

    if not np.isclose(
        cas_level,
        opt_level
    ):

        difference_rows.append({
            "control":
                c,

            "CASCADE_level":
                cas_level,

            "OPT_level":
                opt_level,

            "level_difference":
                (
                    cas_level
                    -
                    opt_level
                ),

            "cost":
                float(
                    g19_costs[c]
                ),

            "base_protection":
                float(
                    g19_protection[c]
                )
        })


print("\n" + "-" * 100)
print("CONTROLS RESPONSIBLE FOR CONFIGURATION DIFFERENCE")
print("-" * 100)

if len(
    difference_rows
) == 0:

    print(
        "CASCADE and OPT configurations are identical."
    )

else:

    g19_difference_df = pd.DataFrame(
        difference_rows
    )

    print(
        g19_difference_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


# ============================================================
# 16. Diagnostic summary
# ============================================================

print("\n" + "-" * 100)
print("DIAGNOSTIC SUMMARY")
print("-" * 100)

print(
    f"CASCADE protection       : "
    f"{g19_cascade['protection']:.6f}"
)

print(
    f"OPT protection           : "
    f"{g19_opt['protection']:.6f}"
)

print(
    f"Absolute protection loss : "
    f"{g19_opt['protection'] - g19_cascade['protection']:.6f}"
)

print(
    f"Optimality gap           : "
    f"{g19_gap:.4f}%"
)

print(
    f"CASCADE unused resource  : "
    f"{cascade_unused:.2f}"
)

print(
    f"OPT unused resource      : "
    f"{opt_unused:.2f}"
)


if len(
    repair_rows
) > 0:

    best_repair = (
        g19_repair_df.iloc[0]
    )

    print(
        f"Best direct-upgrade gain : "
        f"{best_repair['protection_gain']:.6f}"
    )

    print(
        f"Best upgrade control     : "
        f"{best_repair['control']}"
    )

else:

    print(
        "Best direct-upgrade gain : "
        "none feasible"
    )


if len(
    exchange_rows
) > 0:

    best_exchange = (
        g19_exchange_df.iloc[0]
    )

    print(
        f"Best exchange gain       : "
        f"{best_exchange['gain_vs_CASCADE']:.6f}"
    )

    print(
        f"Best exchange            : "
        f"upgrade {best_exchange['upgrade']} / "
        f"degrade {best_exchange['degrade']}"
    )

else:

    print(
        "Best exchange gain       : "
        "none feasible"
    )


# ============================================================
# 17. Final integrity checks
# ============================================================

assert (
    g19_cascade[
        "resource"
    ]
    <=
    g19_budget + 1e-8
), (
    "CASCADE exceeds budget."
)

assert (
    g19_opt[
        "resource"
    ]
    <=
    g19_budget + 1e-8
), (
    "OPT exceeds budget."
)

assert (
    g19_opt[
        "protection"
    ]
    + 1e-10
    >=
    g19_cascade[
        "protection"
    ]
), (
    "Exact optimum unexpectedly below CASCADE."
)

assert np.isclose(
    g19_cascade[
        "protection"
    ],
    0.406045,
    atol=1e-5
), (
    "Recovered CASCADE result does not match "
    "the frozen G19 benchmark."
)

assert np.isclose(
    g19_opt[
        "protection"
    ],
    0.518265,
    atol=1e-5
), (
    "Recovered exact optimum does not match "
    "the frozen G19 benchmark."
)

assert len(
    dependency_rows
) == 11, (
    f"Expected 11 G19 dependency edges, "
    f"found {len(dependency_rows)}."
)


print("\n" + "=" * 100)
print("[PASS] Frozen G19 successfully reproduced.")
print("[PASS] 11 dependency edges verified.")
print("[PASS] Diagnostic integrity checks passed.")
print("[OK] Step 1.12G completed.")
print("=" * 100)

CASCADE — STEP 1.12G: G19 WORST-CASE DIAGNOSTIC

Frozen G19 successfully recovered.
Controls : ['R1', 'R2', 'R3', 'R4', 'R5', 'R6', 'R7', 'R8']
G19 keys : ['ids', 'costs', 'protection', 'graph']
[PASS] G19 structure verified.

----------------------------------------------------------------------------------------------------
RESOURCE CONDITION
----------------------------------------------------------------------------------------------------
Full resource : 84.00
50% budget    : 42.00

----------------------------------------------------------------------------------------------------
CASCADE V1 vs EXACT OPTIMUM
----------------------------------------------------------------------------------------------------
CASCADE protection : 0.406045
OPT protection     : 0.518265
CASCADE resource   : 39.25
OPT resource       : 41.25
CASCADE unused     : 2.75
OPT unused         : 0.75
Optimality gap     : 21.6529%

--------------------------------------------------------------------------------

Step 1.12H — CASCADE V2 bounded repair

In [23]:
# ============================================================
# CASCADE — STEP 1.12H
# CASCADE V2: Bounded Local Repair
#
# Purpose:
#   Address two empirically identified V1 weaknesses:
#   1. stranded resource budget
#   2. greedy path dependence
#
# IMPORTANT:
#   - Does NOT use exact optimization internally.
#   - Exact optimum is used ONLY for external evaluation.
#   - CASCADE V1 remains frozen and unchanged.
# ============================================================

print("=" * 100)
print("CASCADE — STEP 1.12H: CASCADE V2 BOUNDED LOCAL REPAIR")
print("=" * 100)


# ============================================================
# 1. Utility
# ============================================================

LEVEL_ARRAY = np.asarray(
    RAND_LEVELS,
    dtype=float
)


def get_level_index(value):

    return int(
        np.argmin(
            np.abs(
                LEVEL_ARRAY
                -
                float(value)
            )
        )
    )


# ============================================================
# 2. CASCADE V2
# ============================================================

def rand_cascade_v2(
    problem,
    budget,
    max_repair_iterations=100
):

    # --------------------------------------------------------
    # Phase A — Frozen CASCADE V1
    # --------------------------------------------------------

    v1 = rand_cascade(
        problem,
        budget
    )

    X = (
        v1[
            "config"
        ].copy()
    )

    ids = list(
        problem[
            "ids"
        ]
    )

    repair_history = []

    initial_P = rand_protection(
        problem,
        X
    )

    initial_R = rand_resource(
        problem,
        X
    )


    # --------------------------------------------------------
    # Phase B — bounded local improvement
    # --------------------------------------------------------

    for repair_iteration in range(
        1,
        max_repair_iterations + 1
    ):

        current_P = rand_protection(
            problem,
            X
        )

        current_R = rand_resource(
            problem,
            X
        )

        candidate_moves = []


        # ====================================================
        # MOVE TYPE 1:
        # Single one-level upgrade
        # ====================================================

        for c in ids:

            current_level = float(
                X[c]
            )

            idx = get_level_index(
                current_level
            )

            if (
                idx
                >=
                len(LEVEL_ARRAY) - 1
            ):
                continue

            new_level = float(
                LEVEL_ARRAY[
                    idx + 1
                ]
            )

            X_test = (
                X.copy()
            )

            X_test[c] = (
                new_level
            )

            R_test = rand_resource(
                problem,
                X_test
            )

            if (
                R_test
                >
                budget + 1e-12
            ):
                continue

            P_test = rand_protection(
                problem,
                X_test
            )

            gain = (
                P_test
                -
                current_P
            )

            if gain > 1e-12:

                candidate_moves.append({
                    "move_type":
                        "upgrade",

                    "upgrade":
                        c,

                    "degrade":
                        None,

                    "new_config":
                        X_test,

                    "resource":
                        R_test,

                    "protection":
                        P_test,

                    "gain":
                        gain
                })


        # ====================================================
        # MOVE TYPE 2:
        # One-level upgrade + one-level degradation
        # ====================================================

        for up_control in ids:

            up_level = float(
                X[
                    up_control
                ]
            )

            up_idx = get_level_index(
                up_level
            )

            if (
                up_idx
                >=
                len(LEVEL_ARRAY) - 1
            ):
                continue

            new_up_level = float(
                LEVEL_ARRAY[
                    up_idx + 1
                ]
            )


            for down_control in ids:

                if (
                    down_control
                    ==
                    up_control
                ):
                    continue

                down_level = float(
                    X[
                        down_control
                    ]
                )

                down_idx = get_level_index(
                    down_level
                )

                if down_idx <= 0:
                    continue

                new_down_level = float(
                    LEVEL_ARRAY[
                        down_idx - 1
                    ]
                )

                X_test = (
                    X.copy()
                )

                X_test[
                    up_control
                ] = (
                    new_up_level
                )

                X_test[
                    down_control
                ] = (
                    new_down_level
                )

                R_test = rand_resource(
                    problem,
                    X_test
                )

                if (
                    R_test
                    >
                    budget + 1e-12
                ):
                    continue

                P_test = rand_protection(
                    problem,
                    X_test
                )

                gain = (
                    P_test
                    -
                    current_P
                )

                if gain > 1e-12:

                    candidate_moves.append({
                        "move_type":
                            "exchange",

                        "upgrade":
                            up_control,

                        "degrade":
                            down_control,

                        "new_config":
                            X_test,

                        "resource":
                            R_test,

                        "protection":
                            P_test,

                        "gain":
                            gain
                    })


        # ----------------------------------------------------
        # Stop at local optimum
        # ----------------------------------------------------

        if len(
            candidate_moves
        ) == 0:

            break


        # ----------------------------------------------------
        # Select best protection gain
        #
        # Tie breaker:
        # choose lower resource consumption.
        # ----------------------------------------------------

        candidate_moves.sort(
            key=lambda m: (
                -m["gain"],
                m["resource"]
            )
        )

        best_move = (
            candidate_moves[0]
        )


        # ----------------------------------------------------
        # Accept best move
        # ----------------------------------------------------

        X = (
            best_move[
                "new_config"
            ].copy()
        )

        repair_history.append({
            "iteration":
                repair_iteration,

            "move_type":
                best_move[
                    "move_type"
                ],

            "upgrade":
                best_move[
                    "upgrade"
                ],

            "degrade":
                best_move[
                    "degrade"
                ],

            "resource_before":
                current_R,

            "resource_after":
                best_move[
                    "resource"
                ],

            "protection_before":
                current_P,

            "protection_after":
                best_move[
                    "protection"
                ],

            "gain":
                best_move[
                    "gain"
                ]
        })


    # --------------------------------------------------------
    # Final V2 state
    # --------------------------------------------------------

    final_P = rand_protection(
        problem,
        X
    )

    final_R = rand_resource(
        problem,
        X
    )

    return {
        "config":
            X,

        "protection":
            final_P,

        "resource":
            final_R,

        "repair_iterations":
            len(
                repair_history
            ),

        "repair_history":
            repair_history,

        "v1_protection":
            initial_P,

        "v1_resource":
            initial_R
    }


# ============================================================
# 3. Test V2 on frozen G19
# ============================================================

v2_start = time.perf_counter()

g19_v2 = rand_cascade_v2(
    G19,
    g19_budget
)

g19_v2_runtime = (
    time.perf_counter()
    -
    v2_start
)


# ============================================================
# 4. Compare V1, V2, OPT
# ============================================================

v1_gap = (
    (
        g19_opt["protection"]
        -
        g19_cascade["protection"]
    )
    /
    g19_opt["protection"]
    *
    100
)

v2_gap = (
    (
        g19_opt["protection"]
        -
        g19_v2["protection"]
    )
    /
    g19_opt["protection"]
    *
    100
)


print("\n" + "-" * 100)
print("G19: V1 vs V2 vs EXACT OPTIMUM")
print("-" * 100)

print(
    f"V1 protection       : "
    f"{g19_cascade['protection']:.6f}"
)

print(
    f"V2 protection       : "
    f"{g19_v2['protection']:.6f}"
)

print(
    f"OPT protection      : "
    f"{g19_opt['protection']:.6f}"
)

print()

print(
    f"V1 resource         : "
    f"{g19_cascade['resource']:.2f}"
)

print(
    f"V2 resource         : "
    f"{g19_v2['resource']:.2f}"
)

print(
    f"OPT resource        : "
    f"{g19_opt['resource']:.2f}"
)

print()

print(
    f"V1 gap              : "
    f"{v1_gap:.4f}%"
)

print(
    f"V2 gap              : "
    f"{v2_gap:.4f}%"
)

print(
    f"Gap reduction       : "
    f"{v1_gap - v2_gap:.4f} percentage points"
)

print()

print(
    f"V2 repair moves     : "
    f"{g19_v2['repair_iterations']}"
)

print(
    f"V2 runtime          : "
    f"{g19_v2_runtime:.6f} s"
)


# ============================================================
# 5. Configuration comparison
# ============================================================

comparison_rows = []

for c in g19_ids:

    comparison_rows.append({
        "control":
            c,

        "V1":
            g19_cascade[
                "config"
            ][c],

        "V2":
            g19_v2[
                "config"
            ][c],

        "OPT":
            g19_opt[
                "config"
            ][c]
    })

g19_v2_comparison_df = pd.DataFrame(
    comparison_rows
)


print("\n" + "-" * 100)
print("CONFIGURATION COMPARISON")
print("-" * 100)

print(
    g19_v2_comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.2f}"
    )
)


# ============================================================
# 6. Repair history
# ============================================================

print("\n" + "-" * 100)
print("V2 REPAIR HISTORY")
print("-" * 100)

if (
    len(
        g19_v2[
            "repair_history"
        ]
    )
    ==
    0
):

    print(
        "No repair moves were accepted."
    )

else:

    g19_v2_history_df = pd.DataFrame(
        g19_v2[
            "repair_history"
        ]
    )

    print(
        g19_v2_history_df.to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}"
        )
    )


# ============================================================
# 7. Improvement statistics
# ============================================================

absolute_gain = (
    g19_v2["protection"]
    -
    g19_cascade["protection"]
)

relative_gain = (
    absolute_gain
    /
    g19_cascade["protection"]
    *
    100
)

gap_reduction_pct = (
    (
        v1_gap
        -
        v2_gap
    )
    /
    v1_gap
    *
    100
)


print("\n" + "-" * 100)
print("V2 IMPROVEMENT")
print("-" * 100)

print(
    f"Absolute protection gain : "
    f"{absolute_gain:.6f}"
)

print(
    f"Relative protection gain : "
    f"{relative_gain:.4f}%"
)

print(
    f"V1 gap                  : "
    f"{v1_gap:.4f}%"
)

print(
    f"V2 gap                  : "
    f"{v2_gap:.4f}%"
)

print(
    f"Relative gap reduction  : "
    f"{gap_reduction_pct:.2f}%"
)


# ============================================================
# 8. Integrity checks
# ============================================================

assert (
    g19_v2[
        "resource"
    ]
    <=
    g19_budget + 1e-8
), (
    "V2 exceeds budget."
)

assert (
    g19_v2[
        "protection"
    ]
    + 1e-12
    >=
    g19_cascade[
        "protection"
    ]
), (
    "V2 is worse than V1."
)

assert (
    g19_opt[
        "protection"
    ]
    + 1e-10
    >=
    g19_v2[
        "protection"
    ]
), (
    "V2 exceeds exact optimum — check implementation."
)


print("\n" + "=" * 100)
print("[PASS] V2 satisfies resource budget.")
print("[PASS] V2 protection >= V1 protection.")
print("[PASS] V2 protection <= exact optimum.")
print("[OK] Step 1.12H completed.")
print("=" * 100)

CASCADE — STEP 1.12H: CASCADE V2 BOUNDED LOCAL REPAIR

----------------------------------------------------------------------------------------------------
G19: V1 vs V2 vs EXACT OPTIMUM
----------------------------------------------------------------------------------------------------
V1 protection       : 0.406045
V2 protection       : 0.493815
OPT protection      : 0.518265

V1 resource         : 39.25
V2 resource         : 41.00
OPT resource        : 41.25

V1 gap              : 21.6529%
V2 gap              : 4.7175%
Gap reduction       : 16.9354 percentage points

V2 repair moves     : 8
V2 runtime          : 0.009475 s

----------------------------------------------------------------------------------------------------
CONFIGURATION COMPARISON
----------------------------------------------------------------------------------------------------
control   V1   V2  OPT
     R1 0.75 0.00 0.00
     R2 0.00 1.00 1.00
     R3 1.00 0.50 0.00
     R4 0.50 0.00 0.00
     R5 0.00 0.00 0.25


Step 1.12I — Frozen V1 vs V2 vs OPT across all 90 cases

In [24]:
# ============================================================
# CASCADE — STEP 1.12I
# Frozen 90-case evaluation:
# CASCADE V1 vs CASCADE V2 vs Exact OPT
#
# IMPORTANT:
# - Uses SAME 30 frozen DDGs from Step 1.12E
# - Uses SAME 3 resource ratios
# - V2 algorithm is NOT modified
# - Existing exact-optimum results are reused
# ============================================================

print("=" * 105)
print("CASCADE — STEP 1.12I: V1 vs V2 vs OPT — 90-CASE FROZEN BENCHMARK")
print("=" * 105)


# ============================================================
# 1. Integrity checks
# ============================================================

assert "benchmark_problems" in globals(), (
    "benchmark_problems not found."
)

assert "benchmark_df" in globals(), (
    "benchmark_df not found."
)

assert "rand_cascade_v2" in globals(), (
    "rand_cascade_v2 not found. "
    "Run Step 1.12H first."
)

assert len(
    benchmark_problems
) == 30, (
    f"Expected 30 frozen graphs, "
    f"found {len(benchmark_problems)}."
)

assert len(
    benchmark_df
) == 90, (
    f"Expected 90 frozen comparisons, "
    f"found {len(benchmark_df)}."
)


print(
    f"\nFrozen DDGs verified : "
    f"{len(benchmark_problems)}"
)

print(
    f"Frozen cases verified: "
    f"{len(benchmark_df)}"
)


# ============================================================
# 2. Evaluate V1 and V2 on exactly the same cases
# ============================================================

v2_benchmark_rows = []

evaluation_start = time.perf_counter()


for graph_idx in range(
    1,
    31
):

    graph_id = (
        f"G{graph_idx:02d}"
    )

    problem = (
        benchmark_problems[
            graph_id
        ]
    )

    print(
        f"\n[{graph_id}]"
    )


    for ratio in [
        0.80,
        0.65,
        0.50
    ]:

        # ----------------------------------------------------
        # Recover frozen benchmark row
        # ----------------------------------------------------

        frozen_match = benchmark_df[
            (
                benchmark_df[
                    "graph_id"
                ]
                ==
                graph_id
            )
            &
            (
                np.isclose(
                    benchmark_df[
                        "resource_ratio"
                    ],
                    ratio
                )
            )
        ]

        assert len(
            frozen_match
        ) == 1, (
            f"Could not uniquely recover "
            f"{graph_id} ratio={ratio}."
        )

        frozen_row = (
            frozen_match.iloc[0]
        )

        budget = float(
            frozen_row[
                "budget"
            ]
        )

        opt_protection = float(
            frozen_row[
                "opt_protection"
            ]
        )

        opt_resource = float(
            frozen_row[
                "opt_resource"
            ]
        )


        # ----------------------------------------------------
        # CASCADE V1
        # ----------------------------------------------------

        v1_start = (
            time.perf_counter()
        )

        v1 = rand_cascade(
            problem,
            budget
        )

        v1_runtime = (
            time.perf_counter()
            -
            v1_start
        )


        # ----------------------------------------------------
        # CASCADE V2
        # ----------------------------------------------------

        v2_start = (
            time.perf_counter()
        )

        v2 = rand_cascade_v2(
            problem,
            budget
        )

        v2_runtime = (
            time.perf_counter()
            -
            v2_start
        )


        # ----------------------------------------------------
        # Verify V1 reproduces frozen benchmark
        # ----------------------------------------------------

        assert np.isclose(
            v1[
                "protection"
            ],
            frozen_row[
                "cascade_protection"
            ],
            atol=1e-10
        ), (
            f"V1 reproduction mismatch "
            f"for {graph_id}, ratio={ratio}."
        )


        assert np.isclose(
            v1[
                "resource"
            ],
            frozen_row[
                "cascade_resource"
            ],
            atol=1e-10
        ), (
            f"V1 resource mismatch "
            f"for {graph_id}, ratio={ratio}."
        )


        # ----------------------------------------------------
        # Compute gaps
        # ----------------------------------------------------

        v1_gap = (
            (
                opt_protection
                -
                v1[
                    "protection"
                ]
            )
            /
            opt_protection
            *
            100.0
        )

        v2_gap = (
            (
                opt_protection
                -
                v2[
                    "protection"
                ]
            )
            /
            opt_protection
            *
            100.0
        )


        # ----------------------------------------------------
        # Improvement
        # ----------------------------------------------------

        protection_gain = (
            v2[
                "protection"
            ]
            -
            v1[
                "protection"
            ]
        )

        gap_reduction_pp = (
            v1_gap
            -
            v2_gap
        )


        if (
            v1_gap
            >
            1e-12
        ):

            relative_gap_reduction = (
                gap_reduction_pp
                /
                v1_gap
                *
                100.0
            )

        else:

            relative_gap_reduction = (
                0.0
            )


        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        if (
            v2[
                "protection"
            ]
            >
            v1[
                "protection"
            ]
            + 1e-10
        ):

            outcome = (
                "improved"
            )

        elif np.isclose(
            v2[
                "protection"
            ],
            v1[
                "protection"
            ],
            atol=1e-10
        ):

            outcome = (
                "same"
            )

        else:

            outcome = (
                "worse"
            )


        # ----------------------------------------------------
        # Exact-match flags
        # ----------------------------------------------------

        v1_exact = np.isclose(
            v1[
                "protection"
            ],
            opt_protection,
            atol=1e-10
        )

        v2_exact = np.isclose(
            v2[
                "protection"
            ],
            opt_protection,
            atol=1e-10
        )


        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        v2_benchmark_rows.append({

            "graph_id":
                graph_id,

            "resource_ratio":
                ratio,

            "budget":
                budget,

            "edge_count":
                int(
                    frozen_row[
                        "edge_count"
                    ]
                ),

            "v1_protection":
                v1[
                    "protection"
                ],

            "v2_protection":
                v2[
                    "protection"
                ],

            "opt_protection":
                opt_protection,

            "v1_resource":
                v1[
                    "resource"
                ],

            "v2_resource":
                v2[
                    "resource"
                ],

            "opt_resource":
                opt_resource,

            "v1_unused":
                budget
                -
                v1[
                    "resource"
                ],

            "v2_unused":
                budget
                -
                v2[
                    "resource"
                ],

            "opt_unused":
                budget
                -
                opt_resource,

            "v1_gap_pct":
                v1_gap,

            "v2_gap_pct":
                v2_gap,

            "gap_reduction_pp":
                gap_reduction_pp,

            "relative_gap_reduction_pct":
                relative_gap_reduction,

            "protection_gain":
                protection_gain,

            "outcome":
                outcome,

            "v1_exact":
                v1_exact,

            "v2_exact":
                v2_exact,

            "v1_runtime_s":
                v1_runtime,

            "v2_runtime_s":
                v2_runtime,

            "repair_iterations":
                v2[
                    "repair_iterations"
                ]
        })


        print(
            f"  {ratio:.0%} | "
            f"V1={v1['protection']:.6f} | "
            f"V2={v2['protection']:.6f} | "
            f"OPT={opt_protection:.6f} | "
            f"Gap1={v1_gap:.3f}% | "
            f"Gap2={v2_gap:.3f}% | "
            f"{outcome}"
        )


# ============================================================
# 3. DataFrame
# ============================================================

v2_benchmark_df = pd.DataFrame(
    v2_benchmark_rows
)

evaluation_time = (
    time.perf_counter()
    -
    evaluation_start
)


# ============================================================
# 4. Global integrity checks
# ============================================================

assert len(
    v2_benchmark_df
) == 90


# V2 must never violate resource budget
assert (
    v2_benchmark_df[
        "v2_resource"
    ]
    <=
    v2_benchmark_df[
        "budget"
    ]
    + 1e-8
).all(), (
    "V2 resource violation detected."
)


# Local repair accepts only positive moves,
# so V2 must never be worse than V1
assert (
    v2_benchmark_df[
        "v2_protection"
    ]
    + 1e-10
    >=
    v2_benchmark_df[
        "v1_protection"
    ]
).all(), (
    "V2 worse than V1 detected."
)


# Exact optimizer must remain upper bound
assert (
    v2_benchmark_df[
        "opt_protection"
    ]
    + 1e-8
    >=
    v2_benchmark_df[
        "v2_protection"
    ]
).all(), (
    "V2 appears above exact optimum."
)


# ============================================================
# 5. Overall summary
# ============================================================

n_improved = int(
    (
        v2_benchmark_df[
            "outcome"
        ]
        ==
        "improved"
    ).sum()
)

n_same = int(
    (
        v2_benchmark_df[
            "outcome"
        ]
        ==
        "same"
    ).sum()
)

n_worse = int(
    (
        v2_benchmark_df[
            "outcome"
        ]
        ==
        "worse"
    ).sum()
)


v1_exact_count = int(
    v2_benchmark_df[
        "v1_exact"
    ].sum()
)

v2_exact_count = int(
    v2_benchmark_df[
        "v2_exact"
    ].sum()
)


print("\n" + "=" * 105)
print("OVERALL V1 vs V2 SUMMARY")
print("=" * 105)

print(
    f"Evaluation runtime : "
    f"{evaluation_time:.3f} s"
)

print()

print(
    f"Improved cases     : "
    f"{n_improved}/90 "
    f"({100*n_improved/90:.2f}%)"
)

print(
    f"Unchanged cases    : "
    f"{n_same}/90 "
    f"({100*n_same/90:.2f}%)"
)

print(
    f"Worse cases        : "
    f"{n_worse}/90 "
    f"({100*n_worse/90:.2f}%)"
)

print()

print(
    f"V1 mean gap        : "
    f"{v2_benchmark_df['v1_gap_pct'].mean():.4f}%"
)

print(
    f"V2 mean gap        : "
    f"{v2_benchmark_df['v2_gap_pct'].mean():.4f}%"
)

print()

print(
    f"V1 median gap      : "
    f"{v2_benchmark_df['v1_gap_pct'].median():.4f}%"
)

print(
    f"V2 median gap      : "
    f"{v2_benchmark_df['v2_gap_pct'].median():.4f}%"
)

print()

print(
    f"V1 maximum gap     : "
    f"{v2_benchmark_df['v1_gap_pct'].max():.4f}%"
)

print(
    f"V2 maximum gap     : "
    f"{v2_benchmark_df['v2_gap_pct'].max():.4f}%"
)

print()

print(
    f"V1 exact matches   : "
    f"{v1_exact_count}/90 "
    f"({100*v1_exact_count/90:.2f}%)"
)

print(
    f"V2 exact matches   : "
    f"{v2_exact_count}/90 "
    f"({100*v2_exact_count/90:.2f}%)"
)


# ============================================================
# 6. Near-optimality thresholds
# ============================================================

print("\n" + "-" * 105)
print("NEAR-OPTIMALITY THRESHOLDS")
print("-" * 105)

for threshold in [
    1.0,
    2.0,
    5.0,
    10.0
]:

    v1_count = int(
        (
            v2_benchmark_df[
                "v1_gap_pct"
            ]
            <=
            threshold
        ).sum()
    )

    v2_count = int(
        (
            v2_benchmark_df[
                "v2_gap_pct"
            ]
            <=
            threshold
        ).sum()
    )

    print(
        f"Gap <= {threshold:>4.1f}% | "
        f"V1: {v1_count:>2}/90 "
        f"({100*v1_count/90:6.2f}%) | "
        f"V2: {v2_count:>2}/90 "
        f"({100*v2_count/90:6.2f}%)"
    )


# ============================================================
# 7. Summary by resource condition
# ============================================================

summary_by_ratio = (
    v2_benchmark_df
    .groupby(
        "resource_ratio"
    )
    .agg(
        cases=(
            "graph_id",
            "size"
        ),

        v1_mean_gap=(
            "v1_gap_pct",
            "mean"
        ),

        v2_mean_gap=(
            "v2_gap_pct",
            "mean"
        ),

        v1_median_gap=(
            "v1_gap_pct",
            "median"
        ),

        v2_median_gap=(
            "v2_gap_pct",
            "median"
        ),

        v1_max_gap=(
            "v1_gap_pct",
            "max"
        ),

        v2_max_gap=(
            "v2_gap_pct",
            "max"
        ),

        mean_protection_gain=(
            "protection_gain",
            "mean"
        ),

        v1_mean_unused=(
            "v1_unused",
            "mean"
        ),

        v2_mean_unused=(
            "v2_unused",
            "mean"
        ),

        mean_repair_iterations=(
            "repair_iterations",
            "mean"
        ),

        v1_exact_matches=(
            "v1_exact",
            "sum"
        ),

        v2_exact_matches=(
            "v2_exact",
            "sum"
        )
    )
    .reset_index()
)


print("\n" + "-" * 105)
print("RESULTS BY RESOURCE CONDITION")
print("-" * 105)

print(
    summary_by_ratio.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 8. Runtime
# ============================================================

mean_v1_runtime = (
    v2_benchmark_df[
        "v1_runtime_s"
    ].mean()
)

mean_v2_runtime = (
    v2_benchmark_df[
        "v2_runtime_s"
    ].mean()
)

runtime_multiplier = (
    mean_v2_runtime
    /
    mean_v1_runtime
)


print("\n" + "-" * 105)
print("RUNTIME")
print("-" * 105)

print(
    f"Mean V1 runtime     : "
    f"{mean_v1_runtime:.6f} s"
)

print(
    f"Mean V2 runtime     : "
    f"{mean_v2_runtime:.6f} s"
)

print(
    f"V2/V1 multiplier   : "
    f"{runtime_multiplier:.2f}x"
)

print(
    f"Previous exact mean : "
    f"{benchmark_df['exact_runtime_s'].mean():.6f} s"
)

print(
    f"Exact/V2 ratio      : "
    f"{benchmark_df['exact_runtime_s'].mean()/mean_v2_runtime:,.1f}x"
)


# ============================================================
# 9. Worst V2 cases
# ============================================================

worst_v2 = (
    v2_benchmark_df
    .sort_values(
        "v2_gap_pct",
        ascending=False
    )
    .head(10)
)


print("\n" + "-" * 105)
print("TOP 10 V2 OPTIMALITY GAPS")
print("-" * 105)

print(
    worst_v2[
        [
            "graph_id",
            "resource_ratio",
            "v1_protection",
            "v2_protection",
            "opt_protection",
            "v1_gap_pct",
            "v2_gap_pct",
            "gap_reduction_pp",
            "v1_unused",
            "v2_unused",
            "repair_iterations"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 10. Largest V2 improvements
# ============================================================

largest_improvements = (
    v2_benchmark_df
    .sort_values(
        "gap_reduction_pp",
        ascending=False
    )
    .head(10)
)


print("\n" + "-" * 105)
print("TOP 10 GAP REDUCTIONS")
print("-" * 105)

print(
    largest_improvements[
        [
            "graph_id",
            "resource_ratio",
            "v1_gap_pct",
            "v2_gap_pct",
            "gap_reduction_pp",
            "protection_gain",
            "repair_iterations"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 11. Final
# ============================================================

print("\n" + "=" * 105)
print("[PASS] Same 30 frozen DDGs used.")
print("[PASS] Same 90 resource cases used.")
print("[PASS] V1 reproduced frozen results.")
print("[PASS] V2 never violated resource budgets.")
print("[PASS] V2 never performed below V1.")
print("[PASS] Exact optimum remained the upper bound.")
print("[OK] Step 1.12I completed.")
print("=" * 105)

CASCADE — STEP 1.12I: V1 vs V2 vs OPT — 90-CASE FROZEN BENCHMARK

Frozen DDGs verified : 30
Frozen cases verified: 90

[G01]
  80% | V1=0.892794 | V2=0.892794 | OPT=0.892794 | Gap1=0.000% | Gap2=0.000% | same
  65% | V1=0.717323 | V2=0.717323 | OPT=0.717323 | Gap1=0.000% | Gap2=0.000% | same
  50% | V1=0.554197 | V2=0.576209 | OPT=0.576209 | Gap1=3.820% | Gap2=0.000% | improved

[G02]
  80% | V1=0.860963 | V2=0.860963 | OPT=0.860963 | Gap1=0.000% | Gap2=0.000% | same
  65% | V1=0.712541 | V2=0.712541 | OPT=0.715365 | Gap1=0.395% | Gap2=0.395% | same
  50% | V1=0.519449 | V2=0.554942 | OPT=0.561546 | Gap1=7.497% | Gap2=1.176% | improved

[G03]
  80% | V1=0.822869 | V2=0.830545 | OPT=0.830545 | Gap1=0.924% | Gap2=0.000% | improved
  65% | V1=0.717673 | V2=0.717673 | OPT=0.717673 | Gap1=0.000% | Gap2=0.000% | same
  50% | V1=0.578185 | V2=0.583321 | OPT=0.583321 | Gap1=0.881% | Gap2=0.000% | improved

[G04]
  80% | V1=0.873746 | V2=0.873746 | OPT=0.873746 | Gap1=0.000% | Gap2=0.000% | sam

Next: Step 1.12J — statistical validation

In [25]:
# ============================================================
# CASCADE — STEP 1.12J
# Statistical validation of CASCADE V2 improvement
#
# Uses the SAME frozen 90 paired cases from Step 1.12I.
# No algorithm changes.
# ============================================================

print("=" * 100)
print("CASCADE — STEP 1.12J: STATISTICAL VALIDATION OF V1 vs V2")
print("=" * 100)

from scipy import stats


# ============================================================
# 1. Recover paired observations
# ============================================================

v1_gap = v2_benchmark_df["v1_gap_pct"].to_numpy(dtype=float)
v2_gap = v2_benchmark_df["v2_gap_pct"].to_numpy(dtype=float)

v1_prot = v2_benchmark_df["v1_protection"].to_numpy(dtype=float)
v2_prot = v2_benchmark_df["v2_protection"].to_numpy(dtype=float)

gap_reduction = v1_gap - v2_gap
protection_gain = v2_prot - v1_prot

n = len(v1_gap)

assert n == 90
assert np.all(gap_reduction >= -1e-10)
assert np.all(protection_gain >= -1e-10)


# ============================================================
# 2. Descriptive paired statistics
# ============================================================

print("\n" + "-" * 100)
print("1. PAIRED DESCRIPTIVE STATISTICS")
print("-" * 100)

print(f"Paired cases                 : {n}")
print(f"Mean V1 gap                  : {np.mean(v1_gap):.6f}%")
print(f"Mean V2 gap                  : {np.mean(v2_gap):.6f}%")
print(f"Mean gap reduction           : {np.mean(gap_reduction):.6f} pp")
print(f"Median gap reduction         : {np.median(gap_reduction):.6f} pp")
print(f"Mean protection gain         : {np.mean(protection_gain):.6f}")
print(f"Median protection gain       : {np.median(protection_gain):.6f}")

improved = np.sum(protection_gain > 1e-10)
ties = np.sum(np.isclose(protection_gain, 0.0, atol=1e-10))

print(f"Strict improvements          : {improved}/{n}")
print(f"Ties                         : {ties}/{n}")


# ============================================================
# 3. Wilcoxon signed-rank test
#
# Primary paired non-parametric test.
# H0: paired V1 and V2 gaps have no systematic difference.
# ============================================================

wilcoxon_result = stats.wilcoxon(
    v1_gap,
    v2_gap,
    alternative="greater",
    zero_method="wilcox"
)

print("\n" + "-" * 100)
print("2. WILCOXON SIGNED-RANK TEST")
print("-" * 100)

print(f"Statistic                    : {wilcoxon_result.statistic:.6f}")
print(f"One-sided p-value            : {wilcoxon_result.pvalue:.12g}")


# ============================================================
# 4. Sign test
#
# Among non-tied cases, V2 should improve more often
# than expected under p=0.5.
# ============================================================

non_tied = protection_gain[
    np.abs(protection_gain) > 1e-10
]

n_non_tied = len(non_tied)
n_positive = int(np.sum(non_tied > 0))

sign_result = stats.binomtest(
    n_positive,
    n_non_tied,
    p=0.5,
    alternative="greater"
)

print("\n" + "-" * 100)
print("3. EXACT SIGN TEST")
print("-" * 100)

print(f"Non-tied cases               : {n_non_tied}")
print(f"V2 improvements              : {n_positive}")
print(f"V2 degradations              : {n_non_tied - n_positive}")
print(f"One-sided exact p-value      : {sign_result.pvalue:.12g}")


# ============================================================
# 5. Paired effect size
#
# Rank-biserial style dominance measure:
#
# (positive - negative) / non-tied
#
# Because V2 is constructed to never accept worsening moves,
# interpret cautiously; report as descriptive effect size.
# ============================================================

positive_count = int(
    np.sum(gap_reduction > 1e-10)
)

negative_count = int(
    np.sum(gap_reduction < -1e-10)
)

nonzero_count = (
    positive_count
    +
    negative_count
)

if nonzero_count > 0:

    rank_biserial_direction = (
        positive_count
        -
        negative_count
    ) / nonzero_count

else:

    rank_biserial_direction = np.nan


print("\n" + "-" * 100)
print("4. DIRECTIONAL EFFECT SUMMARY")
print("-" * 100)

print(f"Positive gap reductions      : {positive_count}")
print(f"Negative gap reductions      : {negative_count}")
print(f"Zero changes                 : {n - nonzero_count}")
print(f"Directional dominance        : {rank_biserial_direction:.6f}")


# ============================================================
# 6. Bootstrap 95% CI for mean gap reduction
#
# Fixed seed for reproducibility.
# ============================================================

BOOT_SEED = 20260930
BOOT_N = 20000

boot_rng = np.random.default_rng(
    BOOT_SEED
)

boot_means = np.empty(
    BOOT_N,
    dtype=float
)

for b in range(BOOT_N):

    sample_idx = boot_rng.integers(
        0,
        n,
        size=n
    )

    boot_means[b] = np.mean(
        gap_reduction[
            sample_idx
        ]
    )


boot_ci_low = np.percentile(
    boot_means,
    2.5
)

boot_ci_high = np.percentile(
    boot_means,
    97.5
)


print("\n" + "-" * 100)
print("5. BOOTSTRAP 95% CI — MEAN GAP REDUCTION")
print("-" * 100)

print(f"Bootstrap seed               : {BOOT_SEED}")
print(f"Bootstrap samples            : {BOOT_N:,}")
print(f"Mean reduction               : {np.mean(gap_reduction):.6f} pp")
print(f"95% bootstrap CI             : [{boot_ci_low:.6f}, {boot_ci_high:.6f}] pp")


# ============================================================
# 7. Resource-condition-specific statistics
# ============================================================

condition_rows = []

for ratio in sorted(
    v2_benchmark_df["resource_ratio"].unique()
):

    subset = v2_benchmark_df[
        np.isclose(
            v2_benchmark_df["resource_ratio"],
            ratio
        )
    ]

    d = (
        subset["v1_gap_pct"].to_numpy(dtype=float)
        -
        subset["v2_gap_pct"].to_numpy(dtype=float)
    )

    condition_rows.append({
        "resource_ratio":
            ratio,

        "n":
            len(subset),

        "mean_v1_gap":
            subset["v1_gap_pct"].mean(),

        "mean_v2_gap":
            subset["v2_gap_pct"].mean(),

        "mean_reduction_pp":
            np.mean(d),

        "median_reduction_pp":
            np.median(d),

        "improved_cases":
            int(
                np.sum(
                    d > 1e-10
                )
            ),

        "unchanged_cases":
            int(
                np.sum(
                    np.isclose(
                        d,
                        0.0,
                        atol=1e-10
                    )
                )
            )
    })


condition_stats_df = pd.DataFrame(
    condition_rows
)


print("\n" + "-" * 100)
print("6. EFFECT BY RESOURCE CONDITION")
print("-" * 100)

print(
    condition_stats_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 8. Exact-optimum rate difference
# ============================================================

v1_exact = v2_benchmark_df[
    "v1_exact"
].to_numpy(dtype=bool)

v2_exact = v2_benchmark_df[
    "v2_exact"
].to_numpy(dtype=bool)

v1_only = int(
    np.sum(
        v1_exact
        &
        ~v2_exact
    )
)

v2_only = int(
    np.sum(
        ~v1_exact
        &
        v2_exact
    )
)

both_exact = int(
    np.sum(
        v1_exact
        &
        v2_exact
    )
)

neither_exact = int(
    np.sum(
        ~v1_exact
        &
        ~v2_exact
    )
)


# McNemar exact test via binomial on discordant pairs
discordant = (
    v1_only
    +
    v2_only
)

if discordant > 0:

    mcnemar_result = stats.binomtest(
        v2_only,
        discordant,
        p=0.5,
        alternative="greater"
    )

    mcnemar_p = (
        mcnemar_result.pvalue
    )

else:

    mcnemar_p = np.nan


print("\n" + "-" * 100)
print("7. EXACT-OPTIMUM MATCH ANALYSIS")
print("-" * 100)

print(f"Both exact                   : {both_exact}")
print(f"V1 exact only                : {v1_only}")
print(f"V2 exact only                : {v2_only}")
print(f"Neither exact                : {neither_exact}")
print(f"Discordant pairs             : {discordant}")
print(f"Exact McNemar/binomial p     : {mcnemar_p:.12g}")


# ============================================================
# 9. Runtime overhead
# ============================================================

v1_runtime = v2_benchmark_df[
    "v1_runtime_s"
].to_numpy(dtype=float)

v2_runtime = v2_benchmark_df[
    "v2_runtime_s"
].to_numpy(dtype=float)

runtime_difference = (
    v2_runtime
    -
    v1_runtime
)


print("\n" + "-" * 100)
print("8. RUNTIME OVERHEAD")
print("-" * 100)

print(f"Mean V1 runtime              : {np.mean(v1_runtime):.6f} s")
print(f"Mean V2 runtime              : {np.mean(v2_runtime):.6f} s")
print(f"Mean absolute overhead       : {np.mean(runtime_difference):.6f} s")
print(f"Median absolute overhead     : {np.median(runtime_difference):.6f} s")
print(f"Mean runtime multiplier      : {np.mean(v2_runtime)/np.mean(v1_runtime):.4f}x")


# ============================================================
# 10. Frozen statistical summary
# ============================================================

print("\n" + "=" * 100)
print("FROZEN STEP 1.12J SUMMARY")
print("=" * 100)

print(
    f"V1 -> V2 mean gap: "
    f"{np.mean(v1_gap):.4f}% -> "
    f"{np.mean(v2_gap):.4f}%"
)

print(
    f"Mean reduction: "
    f"{np.mean(gap_reduction):.4f} pp "
    f"(95% bootstrap CI "
    f"[{boot_ci_low:.4f}, {boot_ci_high:.4f}])"
)

print(
    f"Wilcoxon one-sided p: "
    f"{wilcoxon_result.pvalue:.6g}"
)

print(
    f"Exact sign-test p: "
    f"{sign_result.pvalue:.6g}"
)

print(
    f"Exact matches: "
    f"{np.sum(v1_exact)}/90 -> "
    f"{np.sum(v2_exact)}/90"
)

print(
    f"Exact-match McNemar p: "
    f"{mcnemar_p:.6g}"
)

print(
    f"Runtime: "
    f"{np.mean(v1_runtime):.6f}s -> "
    f"{np.mean(v2_runtime):.6f}s"
)


# ============================================================
# 11. Integrity
# ============================================================

assert np.all(
    gap_reduction
    >=
    -1e-10
)

assert np.sum(
    v2_exact
) >= np.sum(
    v1_exact
)

print("\n[PASS] Statistical inputs verified.")
print("[PASS] Paired 90-case structure preserved.")
print("[OK] Step 1.12J completed.")

CASCADE — STEP 1.12J: STATISTICAL VALIDATION OF V1 vs V2

----------------------------------------------------------------------------------------------------
1. PAIRED DESCRIPTIVE STATISTICS
----------------------------------------------------------------------------------------------------
Paired cases                 : 90
Mean V1 gap                  : 1.283897%
Mean V2 gap                  : 0.229138%
Mean gap reduction           : 1.054760 pp
Median gap reduction         : 0.020663 pp
Mean protection gain         : 0.006960
Median protection gain       : 0.000142
Strict improvements          : 46/90
Ties                         : 44/90

----------------------------------------------------------------------------------------------------
2. WILCOXON SIGNED-RANK TEST
----------------------------------------------------------------------------------------------------
Statistic                    : 1081.000000
One-sided p-value            : 1.7614752382e-09

---------------------------

Next: Step 1.13 — Scalability and unseen DDGs

In [26]:
# ============================================================
# CASCADE — STEP 1.13A
# UNSEEN HOLDOUT DDG GENERALIZATION TEST
#
# Purpose:
#   Evaluate the FROZEN CASCADE V2 algorithm on completely
#   new random dependency graphs that were NOT used during
#   V2 development.
#
# Design:
#   100 new DDGs
#   8 controls each
#   3 resource conditions: 80%, 65%, 50%
#
# Total:
#   300 unseen CASCADE V1/V2 comparisons
#
# IMPORTANT:
#   - New independent seed
#   - Do NOT modify CASCADE V2 after seeing these results
#   - Exact optimization intentionally deferred to a smaller
#     holdout subset in the next step
# ============================================================

print("=" * 105)
print("CASCADE — STEP 1.13A: UNSEEN HOLDOUT DDG GENERALIZATION")
print("=" * 105)

# ============================================================
# 1. Frozen holdout settings
# ============================================================

HOLDOUT_SEED = 20261001

HOLDOUT_GRAPHS = 100

HOLDOUT_N = 8

HOLDOUT_LEVELS = np.array(
    [0.00, 0.25, 0.50, 0.75, 1.00],
    dtype=float
)

HOLDOUT_EDGE_PROBABILITY = 0.28

HOLDOUT_RESOURCE_RATIOS = [
    0.80,
    0.65,
    0.50
]

holdout_rng = np.random.default_rng(
    HOLDOUT_SEED
)

print(
    f"\nHoldout seed              : "
    f"{HOLDOUT_SEED}"
)

print(
    f"Unseen DDGs               : "
    f"{HOLDOUT_GRAPHS}"
)

print(
    f"Controls per DDG          : "
    f"{HOLDOUT_N}"
)

print(
    f"Edge probability          : "
    f"{HOLDOUT_EDGE_PROBABILITY}"
)

print(
    f"Resource conditions       : "
    f"{HOLDOUT_RESOURCE_RATIOS}"
)

print(
    f"Total unseen comparisons  : "
    f"{HOLDOUT_GRAPHS * len(HOLDOUT_RESOURCE_RATIOS)}"
)


# ============================================================
# 2. Independent random DDG generator
#
# Same distribution as frozen randomized benchmark,
# but NEW independent seed and NEW problem instances.
# ============================================================

def generate_holdout_problem(
    rng,
    n_controls=8,
    edge_probability=0.28
):

    ids = [
        f"R{i+1}"
        for i in range(
            n_controls
        )
    ]

    # --------------------------------------------------------
    # Independent resource costs
    # --------------------------------------------------------

    costs = {
        c: int(
            rng.integers(
                5,
                21
            )
        )
        for c in ids
    }


    # --------------------------------------------------------
    # Independent raw protection values
    # --------------------------------------------------------

    raw_protection = rng.uniform(
        0.03,
        0.20,
        size=n_controls
    )

    raw_protection = (
        raw_protection
        /
        raw_protection.sum()
    )

    protection = {
        c: float(
            raw_protection[i]
        )
        for i, c in enumerate(
            ids
        )
    }


    # --------------------------------------------------------
    # DAG dependency graph
    #
    # Only i < j edges are permitted.
    # --------------------------------------------------------

    graph = {
        c: []
        for c in ids
    }

    edge_count = 0

    for i in range(
        n_controls
    ):

        for j in range(
            i + 1,
            n_controls
        ):

            if (
                rng.random()
                <
                edge_probability
            ):

                weight = float(
                    rng.uniform(
                        0.15,
                        0.85
                    )
                )

                graph[
                    ids[i]
                ].append(
                    (
                        ids[j],
                        weight
                    )
                )

                edge_count += 1


    # --------------------------------------------------------
    # Guarantee at least one dependency edge
    # --------------------------------------------------------

    if edge_count == 0:

        i = int(
            rng.integers(
                0,
                n_controls - 1
            )
        )

        j = int(
            rng.integers(
                i + 1,
                n_controls
            )
        )

        weight = float(
            rng.uniform(
                0.15,
                0.85
            )
        )

        graph[
            ids[i]
        ].append(
            (
                ids[j],
                weight
            )
        )

        edge_count = 1


    return {
        "ids":
            ids,

        "costs":
            costs,

        "protection":
            protection,

        "graph":
            graph,

        "edge_count":
            edge_count
    }


# ============================================================
# 3. Generate all holdout problems FIRST
#
# This freezes the unseen test set before evaluation.
# ============================================================

holdout_problems = {}

for graph_idx in range(
    1,
    HOLDOUT_GRAPHS + 1
):

    graph_id = (
        f"H{graph_idx:03d}"
    )

    holdout_problems[
        graph_id
    ] = generate_holdout_problem(
        holdout_rng,
        n_controls=HOLDOUT_N,
        edge_probability=HOLDOUT_EDGE_PROBABILITY
    )


print(
    "\n[PASS] All holdout DDGs generated "
    "before algorithm evaluation."
)


# ============================================================
# 4. Validate generated holdout set
# ============================================================

edge_counts = []

full_resources = []

for graph_id, problem in holdout_problems.items():

    assert len(
        problem[
            "ids"
        ]
    ) == HOLDOUT_N

    assert (
        problem[
            "edge_count"
        ]
        >=
        1
    )

    edge_counts.append(
        problem[
            "edge_count"
        ]
    )

    X_full = {
        c: 1.0
        for c in problem[
            "ids"
        ]
    }

    full_R = rand_resource(
        problem,
        X_full
    )

    full_P = rand_protection(
        problem,
        X_full
    )

    full_resources.append(
        full_R
    )

    assert np.isclose(
        full_P,
        1.0,
        atol=1e-10
    )


print("\n" + "-" * 105)
print("HOLDOUT GRAPH CHARACTERISTICS")
print("-" * 105)

print(
    f"Mean edge count            : "
    f"{np.mean(edge_counts):.2f}"
)

print(
    f"Min edge count             : "
    f"{np.min(edge_counts)}"
)

print(
    f"Max edge count             : "
    f"{np.max(edge_counts)}"
)

print(
    f"Mean full resource         : "
    f"{np.mean(full_resources):.2f}"
)

print(
    f"Min full resource          : "
    f"{np.min(full_resources):.2f}"
)

print(
    f"Max full resource          : "
    f"{np.max(full_resources):.2f}"
)


# ============================================================
# 5. Evaluate frozen V1 and V2
# ============================================================

holdout_rows = []

holdout_start = (
    time.perf_counter()
)


for graph_idx in range(
    1,
    HOLDOUT_GRAPHS + 1
):

    graph_id = (
        f"H{graph_idx:03d}"
    )

    problem = (
        holdout_problems[
            graph_id
        ]
    )

    X_full = {
        c: 1.0
        for c in problem[
            "ids"
        ]
    }

    full_resource = rand_resource(
        problem,
        X_full
    )


    for ratio in (
        HOLDOUT_RESOURCE_RATIOS
    ):

        budget = (
            full_resource
            *
            ratio
        )


        # ----------------------------------------------------
        # V1
        # ----------------------------------------------------

        v1_start = (
            time.perf_counter()
        )

        v1 = rand_cascade(
            problem,
            budget
        )

        v1_runtime = (
            time.perf_counter()
            -
            v1_start
        )


        # ----------------------------------------------------
        # V2
        # ----------------------------------------------------

        v2_start = (
            time.perf_counter()
        )

        v2 = rand_cascade_v2(
            problem,
            budget
        )

        v2_runtime = (
            time.perf_counter()
            -
            v2_start
        )


        # ----------------------------------------------------
        # Validate
        # ----------------------------------------------------

        assert (
            v1[
                "resource"
            ]
            <=
            budget + 1e-8
        )

        assert (
            v2[
                "resource"
            ]
            <=
            budget + 1e-8
        )

        assert (
            v2[
                "protection"
            ]
            + 1e-10
            >=
            v1[
                "protection"
            ]
        )


        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        gain = (
            v2[
                "protection"
            ]
            -
            v1[
                "protection"
            ]
        )

        holdout_rows.append({

            "graph_id":
                graph_id,

            "resource_ratio":
                ratio,

            "edge_count":
                problem[
                    "edge_count"
                ],

            "full_resource":
                full_resource,

            "budget":
                budget,

            "v1_resource":
                v1[
                    "resource"
                ],

            "v2_resource":
                v2[
                    "resource"
                ],

            "v1_unused":
                budget
                -
                v1[
                    "resource"
                ],

            "v2_unused":
                budget
                -
                v2[
                    "resource"
                ],

            "v1_protection":
                v1[
                    "protection"
                ],

            "v2_protection":
                v2[
                    "protection"
                ],

            "protection_gain":
                gain,

            "v1_runtime_s":
                v1_runtime,

            "v2_runtime_s":
                v2_runtime,

            "repair_iterations":
                v2[
                    "repair_iterations"
                ]
        })


holdout_df = pd.DataFrame(
    holdout_rows
)

holdout_runtime = (
    time.perf_counter()
    -
    holdout_start
)


# ============================================================
# 6. Overall holdout summary
# ============================================================

improved_mask = (
    holdout_df[
        "protection_gain"
    ]
    >
    1e-10
)

tie_mask = np.isclose(
    holdout_df[
        "protection_gain"
    ],
    0.0,
    atol=1e-10
)

worse_mask = (
    holdout_df[
        "protection_gain"
    ]
    <
    -1e-10
)


print("\n" + "=" * 105)
print("UNSEEN HOLDOUT SUMMARY")
print("=" * 105)

print(
    f"Total comparisons          : "
    f"{len(holdout_df)}"
)

print(
    f"Evaluation runtime         : "
    f"{holdout_runtime:.3f} s"
)

print()

print(
    f"V2 improved               : "
    f"{int(improved_mask.sum())}/{len(holdout_df)} "
    f"({100*improved_mask.mean():.2f}%)"
)

print(
    f"V2 unchanged              : "
    f"{int(tie_mask.sum())}/{len(holdout_df)} "
    f"({100*tie_mask.mean():.2f}%)"
)

print(
    f"V2 worse                  : "
    f"{int(worse_mask.sum())}/{len(holdout_df)} "
    f"({100*worse_mask.mean():.2f}%)"
)

print()

print(
    f"Mean V1 protection        : "
    f"{holdout_df['v1_protection'].mean():.6f}"
)

print(
    f"Mean V2 protection        : "
    f"{holdout_df['v2_protection'].mean():.6f}"
)

print(
    f"Mean protection gain      : "
    f"{holdout_df['protection_gain'].mean():.6f}"
)

print(
    f"Median protection gain    : "
    f"{holdout_df['protection_gain'].median():.6f}"
)

print(
    f"Maximum protection gain   : "
    f"{holdout_df['protection_gain'].max():.6f}"
)

print()

print(
    f"Mean V1 unused resource   : "
    f"{holdout_df['v1_unused'].mean():.4f}"
)

print(
    f"Mean V2 unused resource   : "
    f"{holdout_df['v2_unused'].mean():.4f}"
)

print()

print(
    f"Mean V1 runtime           : "
    f"{holdout_df['v1_runtime_s'].mean():.6f} s"
)

print(
    f"Mean V2 runtime           : "
    f"{holdout_df['v2_runtime_s'].mean():.6f} s"
)

print(
    f"Mean V2/V1 runtime ratio  : "
    f"{holdout_df['v2_runtime_s'].mean()/holdout_df['v1_runtime_s'].mean():.3f}x"
)


# ============================================================
# 7. Results by resource condition
# ============================================================

holdout_by_ratio = (
    holdout_df
    .groupby(
        "resource_ratio"
    )
    .agg(

        cases=(
            "graph_id",
            "size"
        ),

        mean_v1_protection=(
            "v1_protection",
            "mean"
        ),

        mean_v2_protection=(
            "v2_protection",
            "mean"
        ),

        mean_gain=(
            "protection_gain",
            "mean"
        ),

        median_gain=(
            "protection_gain",
            "median"
        ),

        max_gain=(
            "protection_gain",
            "max"
        ),

        mean_v1_unused=(
            "v1_unused",
            "mean"
        ),

        mean_v2_unused=(
            "v2_unused",
            "mean"
        ),

        mean_repair_iterations=(
            "repair_iterations",
            "mean"
        )
    )
    .reset_index()
)


print("\n" + "-" * 105)
print("HOLDOUT RESULTS BY RESOURCE CONDITION")
print("-" * 105)

print(
    holdout_by_ratio.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 8. Improvement counts by resource condition
# ============================================================

count_rows = []

for ratio in (
    HOLDOUT_RESOURCE_RATIOS
):

    subset = holdout_df[
        np.isclose(
            holdout_df[
                "resource_ratio"
            ],
            ratio
        )
    ]

    improved = int(
        (
            subset[
                "protection_gain"
            ]
            >
            1e-10
        ).sum()
    )

    unchanged = int(
        np.isclose(
            subset[
                "protection_gain"
            ],
            0.0,
            atol=1e-10
        ).sum()
    )

    count_rows.append({

        "resource_ratio":
            ratio,

        "improved":
            improved,

        "unchanged":
            unchanged,

        "improved_pct":
            100.0
            *
            improved
            /
            len(subset)
    })


holdout_counts_df = pd.DataFrame(
    count_rows
)


print("\n" + "-" * 105)
print("IMPROVEMENT FREQUENCY BY RESOURCE CONDITION")
print("-" * 105)

print(
    holdout_counts_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 9. Largest unseen improvements
# ============================================================

largest_holdout_gains = (
    holdout_df
    .sort_values(
        "protection_gain",
        ascending=False
    )
    .head(15)
)


print("\n" + "-" * 105)
print("TOP 15 UNSEEN V2 PROTECTION GAINS")
print("-" * 105)

print(
    largest_holdout_gains[
        [
            "graph_id",
            "resource_ratio",
            "edge_count",
            "v1_protection",
            "v2_protection",
            "protection_gain",
            "v1_unused",
            "v2_unused",
            "repair_iterations"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 10. Bootstrap CI for unseen mean protection gain
# ============================================================

HOLDOUT_BOOT_SEED = 20261002
HOLDOUT_BOOT_N = 20000

boot_rng = np.random.default_rng(
    HOLDOUT_BOOT_SEED
)

gains = holdout_df[
    "protection_gain"
].to_numpy(
    dtype=float
)

boot_means = np.empty(
    HOLDOUT_BOOT_N,
    dtype=float
)

for b in range(
    HOLDOUT_BOOT_N
):

    idx = boot_rng.integers(
        0,
        len(gains),
        size=len(gains)
    )

    boot_means[b] = np.mean(
        gains[
            idx
        ]
    )


ci_low = np.percentile(
    boot_means,
    2.5
)

ci_high = np.percentile(
    boot_means,
    97.5
)


print("\n" + "-" * 105)
print("BOOTSTRAP 95% CI — UNSEEN MEAN PROTECTION GAIN")
print("-" * 105)

print(
    f"Bootstrap seed            : "
    f"{HOLDOUT_BOOT_SEED}"
)

print(
    f"Bootstrap samples         : "
    f"{HOLDOUT_BOOT_N:,}"
)

print(
    f"Mean protection gain      : "
    f"{np.mean(gains):.6f}"
)

print(
    f"95% bootstrap CI          : "
    f"[{ci_low:.6f}, {ci_high:.6f}]"
)


# ============================================================
# 11. Freeze holdout summary
# ============================================================

print("\n" + "=" * 105)
print("FROZEN STEP 1.13A SUMMARY")
print("=" * 105)

print(
    f"Unseen DDGs               : "
    f"{HOLDOUT_GRAPHS}"
)

print(
    f"Total cases               : "
    f"{len(holdout_df)}"
)

print(
    f"Improved cases            : "
    f"{int(improved_mask.sum())}"
)

print(
    f"Mean V1 protection        : "
    f"{holdout_df['v1_protection'].mean():.6f}"
)

print(
    f"Mean V2 protection        : "
    f"{holdout_df['v2_protection'].mean():.6f}"
)

print(
    f"Mean gain                 : "
    f"{holdout_df['protection_gain'].mean():.6f}"
)

print(
    f"95% CI mean gain          : "
    f"[{ci_low:.6f}, {ci_high:.6f}]"
)

print(
    f"Mean runtime V1/V2        : "
    f"{holdout_df['v1_runtime_s'].mean():.6f}s / "
    f"{holdout_df['v2_runtime_s'].mean():.6f}s"
)


# ============================================================
# 12. Final integrity checks
# ============================================================

assert len(
    holdout_df
) == 300

assert (
    holdout_df[
        "v2_resource"
    ]
    <=
    holdout_df[
        "budget"
    ]
    + 1e-8
).all()

assert (
    holdout_df[
        "v2_protection"
    ]
    + 1e-10
    >=
    holdout_df[
        "v1_protection"
    ]
).all()

assert (
    holdout_df[
        "protection_gain"
    ]
    >=
    -1e-10
).all()


print("\n[PASS] 100 unseen DDGs evaluated.")
print("[PASS] 300 unseen resource-condition cases evaluated.")
print("[PASS] All resource constraints satisfied.")
print("[PASS] Frozen V2 algorithm used without modification.")
print("[OK] Step 1.13A completed.")

CASCADE — STEP 1.13A: UNSEEN HOLDOUT DDG GENERALIZATION

Holdout seed              : 20261001
Unseen DDGs               : 100
Controls per DDG          : 8
Edge probability          : 0.28
Resource conditions       : [0.8, 0.65, 0.5]
Total unseen comparisons  : 300

[PASS] All holdout DDGs generated before algorithm evaluation.

---------------------------------------------------------------------------------------------------------
HOLDOUT GRAPH CHARACTERISTICS
---------------------------------------------------------------------------------------------------------
Mean edge count            : 7.67
Min edge count             : 2
Max edge count             : 12
Mean full resource         : 101.94
Min full resource          : 78.00
Max full resource          : 127.00

UNSEEN HOLDOUT SUMMARY
Total comparisons          : 300
Evaluation runtime         : 1.333 s

V2 improved               : 156/300 (52.00%)
V2 unchanged              : 144/300 (48.00%)
V2 worse                  : 0/300 (0.0

Step 1.13B — Exact validation on a pre-specified unseen subset

In [27]:
# ============================================================
# CASCADE — STEP 1.13B
# EXACT OPTIMALITY VALIDATION ON PRE-SPECIFIED UNSEEN HOLDOUT
#
# Frozen evaluation subset:
#   H001 ... H020
#
# Resource conditions:
#   80%, 65%, 50%
#
# Total exact comparisons:
#   20 graphs x 3 conditions = 60
#
# IMPORTANT:
# - Subset selected by graph ID, NOT by performance.
# - CASCADE V2 remains frozen.
# - No algorithm modification is permitted from these results.
# ============================================================

print("=" * 110)
print("CASCADE — STEP 1.13B: EXACT VALIDATION ON UNSEEN HOLDOUT DDGs")
print("=" * 110)


# ============================================================
# 1. Frozen evaluation specification
# ============================================================

EXACT_HOLDOUT_IDS = [
    f"H{i:03d}"
    for i in range(1, 21)
]

EXACT_HOLDOUT_RATIOS = [
    0.80,
    0.65,
    0.50
]

EXPECTED_CASES = (
    len(EXACT_HOLDOUT_IDS)
    *
    len(EXACT_HOLDOUT_RATIOS)
)


print(
    f"\nPre-specified DDGs         : "
    f"{EXACT_HOLDOUT_IDS[0]} ... "
    f"{EXACT_HOLDOUT_IDS[-1]}"
)

print(
    f"Number of DDGs            : "
    f"{len(EXACT_HOLDOUT_IDS)}"
)

print(
    f"Resource conditions       : "
    f"{EXACT_HOLDOUT_RATIOS}"
)

print(
    f"Expected exact comparisons: "
    f"{EXPECTED_CASES}"
)


# ============================================================
# 2. Integrity checks
# ============================================================

assert "holdout_problems" in globals(), (
    "holdout_problems not found. "
    "Run Step 1.13A first."
)

assert "holdout_df" in globals(), (
    "holdout_df not found. "
    "Run Step 1.13A first."
)

assert "rand_cascade_v2" in globals(), (
    "CASCADE V2 function not found."
)

assert "rand_exact_optimum" in globals(), (
    "Exact optimizer not found."
)

for graph_id in EXACT_HOLDOUT_IDS:

    assert graph_id in holdout_problems, (
        f"{graph_id} missing from frozen holdout set."
    )


print(
    "\n[PASS] Frozen unseen holdout subset verified."
)


# ============================================================
# 3. Exact evaluation
# ============================================================

exact_holdout_rows = []

total_start = time.perf_counter()


for graph_number, graph_id in enumerate(
    EXACT_HOLDOUT_IDS,
    start=1
):

    problem = (
        holdout_problems[
            graph_id
        ]
    )

    print(
        f"\n[{graph_number:02d}/20] {graph_id}"
    )


    for ratio in EXACT_HOLDOUT_RATIOS:

        # ----------------------------------------------------
        # Recover the already-frozen holdout result
        # ----------------------------------------------------

        frozen_match = holdout_df[
            (
                holdout_df[
                    "graph_id"
                ]
                ==
                graph_id
            )
            &
            (
                np.isclose(
                    holdout_df[
                        "resource_ratio"
                    ],
                    ratio
                )
            )
        ]

        assert len(
            frozen_match
        ) == 1, (
            f"Could not uniquely recover "
            f"{graph_id}, ratio={ratio}."
        )

        frozen_row = (
            frozen_match.iloc[0]
        )

        budget = float(
            frozen_row[
                "budget"
            ]
        )


        # ----------------------------------------------------
        # Reproduce V1
        # ----------------------------------------------------

        v1_start = time.perf_counter()

        v1 = rand_cascade(
            problem,
            budget
        )

        v1_runtime = (
            time.perf_counter()
            -
            v1_start
        )


        # ----------------------------------------------------
        # Reproduce frozen V2
        # ----------------------------------------------------

        v2_start = time.perf_counter()

        v2 = rand_cascade_v2(
            problem,
            budget
        )

        v2_runtime = (
            time.perf_counter()
            -
            v2_start
        )


        # ----------------------------------------------------
        # Verify V1/V2 reproduce Step 1.13A
        # ----------------------------------------------------

        assert np.isclose(
            v1[
                "protection"
            ],
            frozen_row[
                "v1_protection"
            ],
            atol=1e-10
        ), (
            f"V1 reproduction mismatch: "
            f"{graph_id}, ratio={ratio}"
        )

        assert np.isclose(
            v2[
                "protection"
            ],
            frozen_row[
                "v2_protection"
            ],
            atol=1e-10
        ), (
            f"V2 reproduction mismatch: "
            f"{graph_id}, ratio={ratio}"
        )


        # ----------------------------------------------------
        # Exact global optimization
        # ----------------------------------------------------

        exact_start = (
            time.perf_counter()
        )

        opt = rand_exact_optimum(
            problem,
            budget
        )

        exact_runtime = (
            time.perf_counter()
            -
            exact_start
        )


        # ----------------------------------------------------
        # Optimality gaps
        # ----------------------------------------------------

        opt_P = float(
            opt[
                "protection"
            ]
        )

        v1_gap = (
            (
                opt_P
                -
                v1[
                    "protection"
                ]
            )
            /
            opt_P
            *
            100.0
        )

        v2_gap = (
            (
                opt_P
                -
                v2[
                    "protection"
                ]
            )
            /
            opt_P
            *
            100.0
        )


        # ----------------------------------------------------
        # Exact flags
        # ----------------------------------------------------

        v1_exact = np.isclose(
            v1[
                "protection"
            ],
            opt_P,
            atol=1e-10
        )

        v2_exact = np.isclose(
            v2[
                "protection"
            ],
            opt_P,
            atol=1e-10
        )


        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        exact_holdout_rows.append({

            "graph_id":
                graph_id,

            "resource_ratio":
                ratio,

            "edge_count":
                int(
                    problem[
                        "edge_count"
                    ]
                ),

            "budget":
                budget,

            "v1_protection":
                float(
                    v1[
                        "protection"
                    ]
                ),

            "v2_protection":
                float(
                    v2[
                        "protection"
                    ]
                ),

            "opt_protection":
                opt_P,

            "v1_resource":
                float(
                    v1[
                        "resource"
                    ]
                ),

            "v2_resource":
                float(
                    v2[
                        "resource"
                    ]
                ),

            "opt_resource":
                float(
                    opt[
                        "resource"
                    ]
                ),

            "v1_unused":
                budget
                -
                float(
                    v1[
                        "resource"
                    ]
                ),

            "v2_unused":
                budget
                -
                float(
                    v2[
                        "resource"
                    ]
                ),

            "opt_unused":
                budget
                -
                float(
                    opt[
                        "resource"
                    ]
                ),

            "v1_gap_pct":
                v1_gap,

            "v2_gap_pct":
                v2_gap,

            "gap_reduction_pp":
                v1_gap
                -
                v2_gap,

            "v1_exact":
                bool(
                    v1_exact
                ),

            "v2_exact":
                bool(
                    v2_exact
                ),

            "repair_iterations":
                int(
                    v2[
                        "repair_iterations"
                    ]
                ),

            "v1_runtime_s":
                v1_runtime,

            "v2_runtime_s":
                v2_runtime,

            "exact_runtime_s":
                exact_runtime,

            "feasible_states":
                opt.get(
                    "feasible_states",
                    np.nan
                )
        })


        print(
            f"  {ratio:.0%} | "
            f"V1={v1['protection']:.6f} | "
            f"V2={v2['protection']:.6f} | "
            f"OPT={opt_P:.6f} | "
            f"Gap1={v1_gap:.3f}% | "
            f"Gap2={v2_gap:.3f}% | "
            f"Exact={bool(v2_exact)} | "
            f"T={exact_runtime:.2f}s"
        )


exact_holdout_df = pd.DataFrame(
    exact_holdout_rows
)

total_runtime = (
    time.perf_counter()
    -
    total_start
)


# ============================================================
# 4. Integrity checks
# ============================================================

assert len(
    exact_holdout_df
) == EXPECTED_CASES

assert (
    exact_holdout_df[
        "v1_resource"
    ]
    <=
    exact_holdout_df[
        "budget"
    ]
    + 1e-8
).all()

assert (
    exact_holdout_df[
        "v2_resource"
    ]
    <=
    exact_holdout_df[
        "budget"
    ]
    + 1e-8
).all()

assert (
    exact_holdout_df[
        "opt_resource"
    ]
    <=
    exact_holdout_df[
        "budget"
    ]
    + 1e-8
).all()

assert (
    exact_holdout_df[
        "opt_protection"
    ]
    + 1e-8
    >=
    exact_holdout_df[
        "v2_protection"
    ]
).all()

assert (
    exact_holdout_df[
        "v2_protection"
    ]
    + 1e-10
    >=
    exact_holdout_df[
        "v1_protection"
    ]
).all()


# ============================================================
# 5. Overall gap statistics
# ============================================================

print("\n" + "=" * 110)
print("UNSEEN EXACT-OPTIMALITY SUMMARY")
print("=" * 110)

print(
    f"Total exact cases          : "
    f"{len(exact_holdout_df)}"
)

print(
    f"Total evaluation time      : "
    f"{total_runtime:.2f} s "
    f"({total_runtime/60:.2f} min)"
)

print()

for version in [
    "v1",
    "v2"
]:

    gaps = (
        exact_holdout_df[
            f"{version}_gap_pct"
        ]
        .to_numpy(
            dtype=float
        )
    )

    print(
        f"{version.upper()} mean gap              : "
        f"{np.mean(gaps):.4f}%"
    )

    print(
        f"{version.upper()} median gap            : "
        f"{np.median(gaps):.4f}%"
    )

    print(
        f"{version.upper()} 90th percentile gap    : "
        f"{np.percentile(gaps, 90):.4f}%"
    )

    print(
        f"{version.upper()} 95th percentile gap    : "
        f"{np.percentile(gaps, 95):.4f}%"
    )

    print(
        f"{version.upper()} maximum gap           : "
        f"{np.max(gaps):.4f}%"
    )

    print()


# ============================================================
# 6. Exact and near-optimal rates
# ============================================================

print("-" * 110)
print("EXACT / NEAR-OPTIMAL RATES")
print("-" * 110)

for threshold in [
    0.0,
    1.0,
    2.0,
    5.0
]:

    if threshold == 0.0:

        v1_count = int(
            exact_holdout_df[
                "v1_exact"
            ].sum()
        )

        v2_count = int(
            exact_holdout_df[
                "v2_exact"
            ].sum()
        )

        label = (
            "Exact optimum"
        )

    else:

        v1_count = int(
            (
                exact_holdout_df[
                    "v1_gap_pct"
                ]
                <=
                threshold
            ).sum()
        )

        v2_count = int(
            (
                exact_holdout_df[
                    "v2_gap_pct"
                ]
                <=
                threshold
            ).sum()
        )

        label = (
            f"Gap <= {threshold:.0f}%"
        )


    print(
        f"{label:<18} | "
        f"V1: {v1_count:>2}/{EXPECTED_CASES} "
        f"({100*v1_count/EXPECTED_CASES:6.2f}%) | "
        f"V2: {v2_count:>2}/{EXPECTED_CASES} "
        f"({100*v2_count/EXPECTED_CASES:6.2f}%)"
    )


# ============================================================
# 7. Resource-condition summary
# ============================================================

ratio_summary = (
    exact_holdout_df
    .groupby(
        "resource_ratio"
    )
    .agg(

        cases=(
            "graph_id",
            "size"
        ),

        v1_mean_gap=(
            "v1_gap_pct",
            "mean"
        ),

        v2_mean_gap=(
            "v2_gap_pct",
            "mean"
        ),

        v1_median_gap=(
            "v1_gap_pct",
            "median"
        ),

        v2_median_gap=(
            "v2_gap_pct",
            "median"
        ),

        v1_max_gap=(
            "v1_gap_pct",
            "max"
        ),

        v2_max_gap=(
            "v2_gap_pct",
            "max"
        ),

        v1_exact_matches=(
            "v1_exact",
            "sum"
        ),

        v2_exact_matches=(
            "v2_exact",
            "sum"
        ),

        mean_gap_reduction=(
            "gap_reduction_pp",
            "mean"
        ),

        mean_v1_unused=(
            "v1_unused",
            "mean"
        ),

        mean_v2_unused=(
            "v2_unused",
            "mean"
        ),

        mean_opt_unused=(
            "opt_unused",
            "mean"
        )
    )
    .reset_index()
)


print("\n" + "-" * 110)
print("RESULTS BY RESOURCE CONDITION")
print("-" * 110)

print(
    ratio_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ============================================================
# 8. Runtime summary
# ============================================================

mean_v1_runtime = (
    exact_holdout_df[
        "v1_runtime_s"
    ].mean()
)

mean_v2_runtime = (
    exact_holdout_df[
        "v2_runtime_s"
    ].mean()
)

mean_exact_runtime = (
    exact_holdout_df[
        "exact_runtime_s"
    ].mean()
)


print("\n" + "-" * 110)
print("RUNTIME SUMMARY")
print("-" * 110)

print(
    f"Mean V1 runtime            : "
    f"{mean_v1_runtime:.6f} s"
)

print(
    f"Mean V2 runtime            : "
    f"{mean_v2_runtime:.6f} s"
)

print(
    f"Mean exact runtime         : "
    f"{mean_exact_runtime:.6f} s"
)

print(
    f"Exact / V2 runtime ratio   : "
    f"{mean_exact_runtime/mean_v2_runtime:,.1f}x"
)


# ============================================================
# 9. Worst unseen V2 cases
# ============================================================

worst_unseen = (
    exact_holdout_df
    .sort_values(
        "v2_gap_pct",
        ascending=False
    )
    .head(10)
)


print("\n" + "-" * 110)
print("TOP 10 UNSEEN V2 OPTIMALITY GAPS")
print("-" * 110)

print(
    worst_unseen[
        [
            "graph_id",
            "resource_ratio",
            "edge_count",
            "v1_protection",
            "v2_protection",
            "opt_protection",
            "v1_gap_pct",
            "v2_gap_pct",
            "gap_reduction_pp",
            "v2_unused",
            "opt_unused",
            "repair_iterations"
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 10. Bootstrap CI for mean V2 optimality gap
# ============================================================

EXACT_BOOT_SEED = 20261003
EXACT_BOOT_N = 20000

exact_boot_rng = np.random.default_rng(
    EXACT_BOOT_SEED
)

v2_gaps = exact_holdout_df[
    "v2_gap_pct"
].to_numpy(
    dtype=float
)

boot_gap_means = np.empty(
    EXACT_BOOT_N,
    dtype=float
)

for b in range(
    EXACT_BOOT_N
):

    idx = exact_boot_rng.integers(
        0,
        len(v2_gaps),
        size=len(v2_gaps)
    )

    boot_gap_means[b] = np.mean(
        v2_gaps[
            idx
        ]
    )


gap_ci_low = np.percentile(
    boot_gap_means,
    2.5
)

gap_ci_high = np.percentile(
    boot_gap_means,
    97.5
)


print("\n" + "-" * 110)
print("BOOTSTRAP 95% CI — MEAN V2 OPTIMALITY GAP")
print("-" * 110)

print(
    f"Seed                       : "
    f"{EXACT_BOOT_SEED}"
)

print(
    f"Samples                    : "
    f"{EXACT_BOOT_N:,}"
)

print(
    f"Mean V2 gap                : "
    f"{np.mean(v2_gaps):.6f}%"
)

print(
    f"95% bootstrap CI           : "
    f"[{gap_ci_low:.6f}, "
    f"{gap_ci_high:.6f}]%"
)


# ============================================================
# 11. Frozen summary
# ============================================================

print("\n" + "=" * 110)
print("FROZEN STEP 1.13B SUMMARY")
print("=" * 110)

print(
    f"Unseen exact cases         : "
    f"{len(exact_holdout_df)}"
)

print(
    f"V1 mean gap                : "
    f"{exact_holdout_df['v1_gap_pct'].mean():.4f}%"
)

print(
    f"V2 mean gap                : "
    f"{exact_holdout_df['v2_gap_pct'].mean():.4f}%"
)

print(
    f"V2 exact matches           : "
    f"{int(exact_holdout_df['v2_exact'].sum())}"
    f"/{len(exact_holdout_df)}"
)

print(
    f"V2 <= 1% gap               : "
    f"{int((exact_holdout_df['v2_gap_pct'] <= 1).sum())}"
    f"/{len(exact_holdout_df)}"
)

print(
    f"V2 <= 2% gap               : "
    f"{int((exact_holdout_df['v2_gap_pct'] <= 2).sum())}"
    f"/{len(exact_holdout_df)}"
)

print(
    f"V2 maximum gap             : "
    f"{exact_holdout_df['v2_gap_pct'].max():.4f}%"
)

print(
    f"Mean V2 gap 95% CI         : "
    f"[{gap_ci_low:.4f}, "
    f"{gap_ci_high:.4f}]%"
)

print(
    f"Mean exact/V2 runtime      : "
    f"{mean_exact_runtime/mean_v2_runtime:,.1f}x"
)


print("\n[PASS] Pre-specified unseen subset preserved.")
print("[PASS] Frozen V2 used without modification.")
print("[PASS] Exact optimum remained the upper bound.")
print("[OK] Step 1.13B completed.")

CASCADE — STEP 1.13B: EXACT VALIDATION ON UNSEEN HOLDOUT DDGs

Pre-specified DDGs         : H001 ... H020
Number of DDGs            : 20
Resource conditions       : [0.8, 0.65, 0.5]
Expected exact comparisons: 60

[PASS] Frozen unseen holdout subset verified.

[01/20] H001
  80% | V1=0.883447 | V2=0.885658 | OPT=0.885658 | Gap1=0.250% | Gap2=0.000% | Exact=True | T=5.95s
  65% | V1=0.771861 | V2=0.774072 | OPT=0.774072 | Gap1=0.286% | Gap2=0.000% | Exact=True | T=5.23s
  50% | V1=0.570648 | V2=0.570648 | OPT=0.570648 | Gap1=0.000% | Gap2=0.000% | Exact=True | T=3.44s

[02/20] H002
  80% | V1=0.711066 | V2=0.736694 | OPT=0.740114 | Gap1=3.925% | Gap2=0.462% | Exact=False | T=5.49s
  65% | V1=0.578928 | V2=0.580141 | OPT=0.595020 | Gap1=2.704% | Gap2=2.501% | Exact=False | T=5.05s
  50% | V1=0.492552 | V2=0.512727 | OPT=0.512727 | Gap1=3.935% | Gap2=0.000% | Exact=True | T=3.27s

[03/20] H003
  80% | V1=0.856177 | V2=0.856177 | OPT=0.856177 | Gap1=0.000% | Gap2=0.000% | Exact=True | T=5.